In [1]:
import pandas as pd

# 1. Load the dataset with low_memory=False to handle mixed types warning
df = pd.read_csv('~/Desktop/SSC_merged_no_blank_keys.csv', low_memory=False)

# 2. Define the mapping using the EXACT raw strings from your index list
column_mapping = {
    "Project key": "project_key",
    # Demographic
    "Age at study visit (automatic calculated field):    Âge au moment de la visite d'étude (champ calculé automatiquement):": "age_at_visit",
    "1. Gender/Genre:": "gender",
    # Clinical
    "What is the current duration of disease? (years)    Quelle est la durée actuelle de la maladie? (years)": "disease_duration",
    "What is the age at diagnosis? (years)    Quel est l'âge du diagnostic? (années)": "age_at_diagnosis",
    "9. What symptoms are currently present?    9. Quels sont les symptômes présents actuellement?": "current_symptoms",
    "8. When the symptoms began, did the symptoms present on one side or both sides of your body?    8. Les symptômes sont-ils apparus d'un seul côté ou des deux côté de votre corps?": "symptom_laterality",
    "8. Do you feel or have people told you that your sense of smell is getting worse?    8. Avez-vous remarqué ou vous a-t-on fait remarquer que vous aviez une perte d'odorat?": "smell_loss",
    "6. What is your dominant hand?    6. Quelle est votre main dominante?": "dominant_hand",
    "20. Do you exercise on a regular basis?    20. Faites-vous de l\'exercice régulièrement?": "exercise",
    "23. Do you currently drink coffee?    23. Buvez-vous du café régulièrement?": "coffee_drinker",
    # MoCA
    "TOTAL SCORE (make sure to include extra point for 12 years or less of education):    SCORE TOTAL (assurez-vous d'inclure un point supplémentaire pour 12 ans ou moins d'éducation) :.x": "moca_total",
    # BDI-II
    "BDI-II Total Score": "bdi_total",
    # MBI-C (Note: using the version with the .x suffix found in your list)
    "MBI-C Grand Total.x": "mbi_total",
    # EHI
    "EHI Handedness Score": "ehi_score",
    # Outcome
    "Enrolment Group:    Groupe d'inscription:": "outcome"
}


# 3. Filter and Rename
# We use a list comprehension to ensure we only grab keys that actually exist in the dataframe
available_cols = [col for col in column_mapping.keys() if col in df.columns]
df_filtered = df[available_cols].copy()
df_filtered.rename(columns=column_mapping, inplace=True)

# 4. Clean the Outcome Variable
# This removes leading/trailing spaces and ensures consistent categories
df_filtered['outcome'] = df_filtered['outcome'].str.strip()

# 5. Review the results
print(f"Successfully extracted {len(df_filtered.columns)} variables.")
print(df_filtered['outcome'].value_counts())
print(df_filtered.head())

df_filtered.to_csv('processed_ssc_data.csv', index=False)

df['Project key'].value_counts().head()


Successfully extracted 16 variables.
outcome
PD (Parkinson's Disease)/(Maladie de Parkinson)        2852
Healthy control/Contrôle                                410
AP (Atypical Parkinsonism)/(Parkinsonisme Atypique)     171
Name: count, dtype: int64
  project_key  age_at_visit          gender  disease_duration  \
0     SSC0001          50.0   Male/Masculin              14.0   
1     SSC0002          65.0  Female/Féminin              11.3   
2     SSC0003          66.0   Male/Masculin              19.6   
3     SSC0004          53.0   Male/Masculin               9.8   
4     SSC0005          52.0   Male/Masculin               9.6   

   age_at_diagnosis                                   current_symptoms  \
0              45.0  Tremor/Tremblements,Muscle stiffness (Rigidity...   
1              62.0  Tremor/Tremblements,Muscle stiffness (Rigidity...   
2              54.0  Tremor/Tremblements,Muscle stiffness (Rigidity...   
3              50.0  Tremor/Tremblements,Muscle stiffness (Rig

Project key
SSC0001    1
SSC2367    1
SSC2356    1
SSC2357    1
SSC2358    1
Name: count, dtype: int64

In [4]:
import torch
import torch.nn as nn

class TabularTransformer(nn.Module):
    def __init__(self, num_categories, num_numerical, embed_dim=32, n_heads=4, n_layers=3, num_classes=3):
        super(TabularTransformer, self).__init__()
        
        # 1. Embeddings for categorical features (Survey items)
        # We assume each survey item has roughly 5 levels (e.g., Likert 1-4 + NA)
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(6, embed_dim) for _ in range(num_categories)
        ])
        
        # 2. Projection for numerical features (Total scores)
        self.num_projection = nn.Linear(num_numerical, embed_dim)
        
        # 3. Transformer Encoder Layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=n_heads, 
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 4. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * (num_categories + 1), 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) # Outputs: PD, Atypical, Control
        )

    def forward(self, x_cat, x_num):
        # x_cat: [Batch, num_categories]
        # x_num: [Batch, num_numerical]
        
        # Embed categorical features
        embeddings = [embed(x_cat[:, i]) for i, embed in enumerate(self.cat_embeddings)]
        x = torch.stack(embeddings, dim=1) # [Batch, num_cat, embed_dim]
        
        # Project numerical features as a single "token"
        num_feat = self.num_projection(x_num).unsqueeze(1) # [Batch, 1, embed_dim]
        
        # Combine all tokens
        x = torch.cat([x, num_feat], dim=1) # [Batch, total_tokens, embed_dim]
        
        # Pass through Transformer
        attn_out = self.transformer(x)
        
        # Flatten and Classify
        out = attn_out.reshape(attn_out.size(0), -1)
        return self.classifier(out)
    

In [8]:
import pandas as pd
from functools import reduce

# 1. Load the Excel file. 
# Setting sheet_name=None reads ALL sheets into a dictionary: {sheet_name: dataframe}
excel_file = 'Case Study/SSC_Full.xlsx'
all_sheets = pd.read_excel(excel_file, sheet_name=None)

# 2. Get a list of all dataframes from the dictionary
dfs = list(all_sheets.values())

# 3. Successively merge all dataframes on the 'Project key' column
# We use 'outer' to keep all participants from all sheets
df_merged = reduce(lambda left, right: pd.merge(
    left, 
    right, 
    on='Project key', 
    how='outer', 
    suffixes=('', '_drop') # Handles duplicate columns
), dfs)

# 4. Optional: Remove duplicate columns if they were created during the merge
# (e.g., if 'Event Name' appeared in every sheet)
df_merged = df_merged.filter(regex='^(?!.*_drop)')

# 5. Save the final result to CSV
df_merged.to_csv('merged_output.csv', index=False)

print(f"Merge complete! Final shape: {df_merged.shape}")


Merge complete! Final shape: (3541, 1127)


In [11]:
df_merged.to_csv('Case Study/SSC_merged.csv', index=False)


In [14]:
cols = list(df_merged.columns)
print(cols)


['Project key', 'Event Name', 'Questionnaire completed by:    Questionnaire complété par : ', 'Questionnaire completed:     Questionnaire rempli: ', 'Site:', 'QPN/RPQ Site:', 'What language does the participant prefer to be contacted in?    Dans quelle langue le participant préfère-t-il être contacté ?', '**FOR NATIONAL COORDINATOR USE ONLY**  Was the participant imported from CaPRI REDCap?', 'External ID: ', 'LARA ID: ', 'Linked PD-MCI id:  ', 'Linked PD-MCI-TMS id:', 'Linked MIGUT id:  ', 'Is this a test participant?', 'The questionnaire was completed by:     Le questionnaire a été complété par: ', 'Does the participant have follow-ups prior to this entry in REDCap (QPN patients only)?     Le participant a-t-il des suivis avant cette inscription dans REDCap (seulement pour les patients RPQ)?', 'How many follow-ups were completed by this participant?     Combien de suivis ont été faits pour ce participant?  ', 'When was the last follow-up (calculate the number of months)?    À quand r

['Project key', 'Event Name', 'Questionnaire completed by:    Questionnaire complété par : ', 'Questionnaire completed:     Questionnaire rempli: ', 'Site:', 'QPN/RPQ Site:', 'What language does the participant prefer to be contacted in?    Dans quelle langue le participant préfère-t-il être contacté ?', '**FOR NATIONAL COORDINATOR USE ONLY**  Was the participant imported from CaPRI REDCap?', 'External ID: ', 'LARA ID: ', 'Linked PD-MCI id:  ', 'Linked PD-MCI-TMS id:', 'Linked MIGUT id:  ', 'Is this a test participant?', 'The questionnaire was completed by:     Le questionnaire a été complété par: ', 'Does the participant have follow-ups prior to this entry in REDCap (QPN patients only)?     Le participant a-t-il des suivis avant cette inscription dans REDCap (seulement pour les patients RPQ)?', 'How many follow-ups were completed by this participant?     Combien de suivis ont été faits pour ce participant?  ', 'When was the last follow-up (calculate the number of months)?    À quand remonte le dernier suivi (calculez le nombre de mois)?', "Date of enrollment (Date participant signed ICF):    Date d'inscription (Date de signature du FC par le participant):", "Date of study visit:     Date de visite d'étude:     *Enter date that epidemiological questionnaire was completed/  Entrez la date à laquelle le questionnaire épidémiologique a été rempli*", 'Date of blood draw:     Date de prise de sang: ', 'Age at blood draw: ', 'Language that the consent form was completed:    Langue dans laquelle le formulaire de consentement a été complété:  ', "Enrolment Group:    Groupe d'inscription:", "Study Status:    Statut dans l'étude:", 'Reason for withdrawal:    Raison pour le retrait:  ', "  'Please note that the participant can withdraw at any time and for any reason. They are not obligated in any way to tell us the reason for withdrawal, but please ask and check the appropriate box should they have shared the withdrawal reason with you.' ", 'If the participant has moved to a city where a C-OPN site is located, please specify the site.     Si le participant a déménagé dans une ville ou se trouve un cite de C-OPN, veuillez preciser le site. ', "If the reason is 'Other', please specify.    Si la raison est 'Autres' veuillez préciser.", 'Date of withdrawal:    Date de retrait:  ', "Reason for exclusion:    Raison pour l'exclusion:", "Date of exclusion:    Date d'exclusion:", 'Registry Status:    Statut dans le registre:', 'Is this participant only taking part in the online questionnaire portion of C-OPN (e.g. questionnaires that can be sent online via REDCap like the demographic and epidemiological questionnaire)?    Ce participant participe-t-il uniquement aux questionnair', 'Recruitment Source:     Source de recrutement: ', 'Specify if you selected other:     Précisez si vous avez sélectionné autre:', 'If hospital was selected for CaPRI, was the participant recruited through the Movement Disorders Clinic (MDC) or the General Neurology Clinic (GNC)?', 'Has the participant participated in research or clinical trials before?    Le/la participant(e) a-t-il/elle déjà participé à des recherches ou à des essais cliniques?', 'Notes:', 'Complete?', 'Questionnaire completed:     Questionnaire rempli:  ', 'How is the questionnaire completed?     Comment le questionnaire est-il rempli?  ', "Age at study visit (automatic calculated field):    Âge au moment de la visite d'étude (champ calculé automatiquement):", '**OTTAWA HOSPITAL & UBC ONLY**  Age at study visit (input manually): ', '1. Gender/Genre:', 'If other, please specify:    Veuillez préciser si vous avez répondu autre:', '2. What is your current marital status?    2. Quel est votre état matrimonial/civil actuel?', '3. What is your current living situation?    3. Quelle est votre situation de ménage actuelle?', "4. What is the highest level of education you have completed?    4. Quel est le plus haut niveau d'étude que vous avez complété? ", "5. Years of education (calculate the total number of years starting from grade 1 onwards including high school and post-secondary (College, University, or others)):    5. Nombre d'années d'études (calculez le nombre total d'années à partir de la première ", 'What is your current source of income?', 'Please specify other income source:', "6. What is your current employment status?    6. Quelle est votre situation d'emploi?", '7. Do you have a regular caregiver? A caregiver as part of this questionnaire is defined as someone that helps you manage day to day living activities (this includes your spouse or partner if applicable).    7. Avez-vous un aidant régulier?  Un aidant dan', '7a. Please specify regular caregiver:    7a. Veuillez préciser qui est votre aidant régulier:                                    ', 'Please specify other caregiver:    Veuillez préciser si vous avez sélectionné autre aidant: ', "7b. Professional status of the caregiver:     7b Situation professionnelle de l'aidant :", "8. If the questionnaire was done in-person, is the regular caregiver present during the visit?    8. Si le questionnaire a été rempli en personne, l'aidant régulier est-il présent lors de la visite?", '9. Do you attend a support group for your disease?    9. Êtes-vous impliqué dans un groupe de soutien pour votre maladie?', '10. Your involvement in a support group is ...    10. Votre implication dans un groupe de soutien est... ', '11. What is your first language?    11. Quelle est votre langue maternelle?', '11a. What is your first language?    11a. Quelle est votre langue maternelle?', '11b. How many language(s) do you speak?    11b, Combien de langues parlez-vous: ', '11c. List of spoken language(s):     11c. Liste de(s) langue(s) parlée(s): ', 'Which of the following best describes your functional level of the English language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue anglaise?', 'Which of the following best describes your functional level of the French language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue française?', 'Which of the following best describes your functional level of the Spanish language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue espagnole?', 'Which of the following best describes your functional level of the Mandarin language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue mandarine?', 'Which of the following best describes your functional level of the Cantonese language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue cantonaise?', 'Which of the following best describes your functional level of the Punjabi language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue punjabi?', 'Which of the following best describes your functional level of the Hindi language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue hindi?', 'Which of the following best describes your functional level of the Arabic language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue arabe?', 'Which of the following best describes your functional level of the Bengali language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue bengali?', 'Which of the following best describes your functional level of the Russian language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue russe?', 'Which of the following best describes your functional level of the Portuguese language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue portugaise?', 'Which of the following best describes your functional level of the Indonesian language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue indonésienne?', 'Which of the following best describes your functional level of the Italian language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue italienne?', 'Which of the following best describes your functional level of the German language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue allemande?', 'Which of the following best describes your functional level of the Filipino language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue philippine?', 'Which of the following best describes your functional level of the Cree languages?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel des langues cries?', 'Which of the following best describes your functional level of the Inuktitut language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue Inuktitut?', 'Which of the following best describes your functional level of the Ojibway language?    Lequel des énoncés suivants décrit le mieux votre niveau fonctionnel de la langue Ojibwé?', 'If you selected other, please specify:     Si vous en avez sélectionné autre, veuillez préciser:', "12. Do you understand English well enough to participate in the study?    12. Comprenez-vous suffisamment l'anglais pour participer à l'étude?", "13. Do you understand French well enough to participate in the study?    13. Comprenez-vous suffisamment le français pour participer à l'étude?", 'If applicable, who helped you with translation?    Si applicable, qui vous a aidé avec la traduction?  ', "Determined diagnosis:  If score = 0, Parkinson's Disease (PD)  If score = 1, Progressive Supranuclear Palsy (PSP)  If score = 2, Multiple System Atrophy (MSA)  If score = 3, Corticobasal Syndrome (CBS)  If score = 4, Dementia with Lewy Bodies (DLB)  If sc", 'Questionnaire completed by:    Questionnaire complété par:', 'Date Correlation :    Concordance avec la date :', 'How is the questionnaire completed?     Comment le questionnaire est-il rempli?', 'A. Weight of the participant (lbs):     A. Poids du participant (lbs):', 'Weight of the participant (kg):     Poids du participant (kg):', 'B. Height of the participant (ft):     B. Taille du participant (ft): ', 'Height of the participant (cm):     Taille du participant (cm): ', 'BMI', 'IMC', "1. Was the patient diagnosed with Parkinson's disease?     1. Est-ce que le patient a reçu un diagnostic de maladie de Parkinson? ", "1a. If 'No' or 'Uncertain', is the diagnosis....    1a. Si la réponse est > ou >, le diagnostic est-il.... ", 'If other, please specify:     Veuillez préciser si autre:', '1b. If applicable, what is the level of certainty of the diagnosis?     1b. Si possible, quel est le niveau de certitude du diagnostic?', '1c. How was the diagnosis confirmed for C-OPN?    1c. Comment le diagnostic a-t-il été confirmé pour RPCO?', 'If other, please specify:     Veuillez préciser si autre:2', '2. What is the date of diagnosis?    2. Quelle est la date de diagnostic? ', 'What is the current duration of disease? (years)    Quelle est la durée actuelle de la maladie? (years)', 'What is the duration of disease at the time the clinical questionnaire was completed? (years)    Quelle est la durée actuelle de la maladie au moment où le questionnaire clinique a été rempli? (years)', 'What is the duration of disease at the time the covid questionnaire was completed? (years)    Quelle est la durée actuelle de la maladie au moment où le questionnaire covid a été rempli? (years)', 'What is the duration of disease at the time the blood draw was completed? (years)    Quelle est la durée actuelle de la maladie au moment au moment du prélèvement sanguin? (years)', "What is the age at diagnosis? (years)    Quel est l'âge du diagnostic? (années)", "**UBC and TOH ONLY**  What is the age at diagnosis? (years)    Quel est l'âge du diagnostic? (années)", '3. Contact with a nurse specializing in parkinsonisms since the last clinical follow-up appointment?    3. Contact avec une infirmière spécialisée dans la parkinsonisme depuis le dernier RDV de suivi clinique ?', '4. What were the first symptoms?    4. Quels étaient les premiers symptômes de la maladie?                   ', 'Please specify other symptoms:    Veuillez préciser quels autres symptômes : ', "5. Date when the symptoms first appear:     5. Date d'apparition des premiers symptômes :", "Current duration since symptoms first appear: (years)     Current durée depuis l'apparition des premiers symptômes: (years)", "Duration since symptoms first appear when the clinical questionnaire was completed: (years)     Durée depuis l'apparition des premiers symptômes lorsque le questionnaire clinique a été rempli:  (years)", "Duration since symptoms first appear when the blood draw was completed: (years)     Durée depuis l'apparition des premiers symptômes au moment du prélèvement sanguin:  (years)", "What is the age at onset of disease? (years)    Quel est l'âge d'apparition de la maladie ? (années)", "**UBC and TOH ONLY**  What is the age at onset of Parkinson's disease? (years)    Quel est l'âge d'apparition de la maladie de Parkinson? (années)", '6. What is your dominant hand?    6. Quelle est votre main dominante?', '7. Are you a converted left-hander?    7. Êtes-vous gaucher contrarié?', "8. When the symptoms began, did the symptoms present on one side or both sides of your body?    8. Les symptômes sont-ils apparus d'un seul côté ou des deux côté de votre corps? ", '9. What symptoms are currently present?    9. Quels sont les symptômes présents actuellement?                              ', 'Please specify other symptoms:    Veuillez préciser quels autres symptômes: ', '10. Do the symptoms affect both sides of your body now?     10. Les symptômes affectent-ils désormais les deux côtés de votre corps?', '11. Does the development of motor symptoms progress gradually?     11. Le développement des symptômes moteurs évolue-t-il de manière progressive ? ', "12. Has there been almost complete remission since the onset of symptoms?    12. Y-a-t-il eu une rémission quasi complète depuis l'apparition des premiers symptômes ?", '13. Do you currently have dyskinesia? (involuntary movements such as writhing, wriggling, twisting, head bobbing, body swaying)    13. Le patient souffre-t-il actuellement de dyskinésies? (mouvement involontaire tels que gigotement, torsion, tordage, mouv', "13a. What impact do the dyskinesia have on your daily activities?    13a. Quel est l'impact des dyskinésies sur vos activités quotidiennes?", '14. Do you experience freezing of gait?   (sudden, brief inability to start or continue rhythmic movements e.g. walking, writing, finger-tapping...)    14. Ressentez-vous du blocage moteur?   (incapacité soudaine et brève de commencer ou de continuer des ', "14a. What impact does the freezing of gait have on your daily activities?    14a. Quel est l'impact du blocage moteur sur vos activités quotidiennes?", '15. Have you fallen in the last 3 months?    15. Avez-vous fait des chutes dans les trois derniers mois?', '15a. How many times did you fall?    15a. Combien de fois avez-vous chuté?', '16. Can you stand unaided?    16. Pouvez-vous vous tenir debout sans aide?', "17. Can you engage in activities outside of the home unaccompanied?    17. Pouvez-vous faire des activités à l'extérieur de la maison sans accompagnement? ", "18. Do you experience motor fluctuations? (On/Off periods)  Good effect from your medications throughout your awake hours mean 'ON' time. If you have some periods of wearing off, low time, bad time, slow time or shaking time, these low periods are 'OFF' t", '18a. What impact do the motor fluctuations (On/Off periods) have on your daily activities?    18a. Quel impact ont les fluctuations motrices (Période On/Off) sur vos activités quotidiennes?   ', "19. Was the patient ON or OFF during the study visit?    19. Le patient était-il ON ou OFF durant la visite d'étude?    *Select 'Not Applicable' if the questionnaire is not completed in-person/Sélectionnez 'Sans objet' si le questionnaire n'est pas rempli", '20. Do you think your medications are working well currently (significant reduction in symptoms at the time of the study visit or during the call with the coordinator)?    20. Pensez-vous que vos médicaments fonctionnent bien présentement (réduction signi', "21. Hoehn & Yahr Rating Scale:    21. Échelle d'évaluation Hoehn & Yahr:", "When was the Hoehn & Yahr evaluation completed?    Quand l'échelle d'évaluation Hoehn & Yahr a-t-il été fait?", "What method did you use to estimate the Hoehn & Yahr Scale?     Quelle méthode avez-vous utilisée pour estimer l'échelle Hoehn & Yahr?", '22. Does the patient have dementia?    22. Est-ce que le patient est atteint de démence?', '22a. Date dementia was diagnosed:    22a. Date du diagnostic de démence:  ', 'Current duration of dementia: (years)    Durée actuelle de la démence: (years)', 'Duration of dementia when the clinical questionnaire was completed: (years)    Durée de la démence lorsque le questionnaire clinique a été rempli: (years)', 'Duration of dementia when the blood draw was completed: (years)    Durée de la démence au moment du prélèvement sanguin:  (years)', "23. Has the patient had surgery for Parkinson's?    23. Le patient a-t-il/elle eu une ou des chirurgies pour la maladie de Parkinson?  ", '23a. Please specify type of surgery:    23a. Veuillez préciser le type de chirurgie:', 'Please specify other surgery:    Veuillez préciser si autre chirurgie:', '23b. Date of DBS surgery:', 'Current duration of DBS surgery: (years) ', 'Duration of DBS surgery when the clinical questionnaire was completed: (years) ', 'Duration of DBS surgery when the blood draw was completed: (years) ', '24. Other symptoms - neurological signs / medical record    24. Autres symptômes - signes neurologiques / dossier médical ', 'Please specify other:     Veuillez préciser autre:', 'Additional Comments:  Commentaires supplémentaires:', '1. Select all current medications:    1. Sélectionnez vos médicaments actuels :', '1a. IR levodopa dosage (Sinemet):    1a. Dose IR lévodopa (Sinemet):   ', '1b. IR levodopa dosage (Prolopa):    1b. Dose IR lévodopa (Prolopa): ', '1c. IR levodopa dosage (Stalevo):    1c. Dose IR lévodopa (Stalevo):', '1d. Intestinal levodopa dosage (Duodopa):    1d. Dose intestinal lévodopa (Duodopa): ', '1e. CR levodopa dosage (Sinemet CR):    1e. Dose CR lévodopa (Sinemet CR):', '1f. ER levodopa dosage (Rytary):    1f. Dose ER lévodopa (Rytary): ', "1g. Name of other PD medication:    1g. Nom d'autres médicaments pour la maladie de Parkinson: ", '1h. Levodopa dosage (Other PD medication):    1h. Dose lévodopa (autre médicament PD):', '1i. Foslevodopa, foscarbidopa subcutaneous infusion (Vyalev)    1i. Foslevodopa, foscarbidopa en perfusion sous-cutanée (Vyalev)', "2. Has the patient shown a significant reduction in symptoms after starting a dopaminergic agent?    2. Réduction significative des symptômes consécutive à la prise d'un agent dopaminergique? ", '2a. Is this improvement in symptoms due to treatment still present?    2a. Cette amélioration des symptômes grâce au traitement est-elle toujours présente?', "2b. If 'no or uncertain', did the medication ever help in the past?    2b. Si non ou incertain, avez-vous déjà pris des médicaments dans le passé qui ont améliorés vos symptômes? ", '2c. How long did the medication help?    2c. Combien de temps le médicament a-t-il aidé?', '3. Current COMT Inhibitor Medications:(Catechol-O-Methyl Transferase)    3. Médication actuelle - Inhibiteur de la COMT: (Catéchol-O-Méthyltransférase)  ', '3a. COMT Inhibitor dosage (entacapone/Comtan):    3a. Dose inhibiteur de la COMT(entacapone/Comtan):', '3b. COMT Inhibitor dosage (entacapone/Stalevo):    3b. Dose inhibiteur de la COMT (entacapone/Stalevo):', '3c. COMT Inhibitor dosage (tolcapone/Tasmar):    3c. Dose inhibiteur de la COMT(tolcapone/Tasmar):', "3d. Name of other COMT Inhibitor:    3d. Nom d'un autre inhibiteur de la COMT:", "3e. COMT Inhibitor dosage (Other):    3e. Dose d'inhibiteur de la COMT (autre):", '4. Current Dopamine Agonist Medications:    4. Médication actuelle - Agoniste dopaminergique:', '4a. Dopamine Agonist dosage (pramipexole/Mirapex):    4a. Dose agoniste dopaminergique (pramipexole/Mirapex):', '4b. Dopamine Agonists dosage (ropinirole/Requip):    4b. Dose agoniste dopaminergique (ropinirole/Requip):', '4c. Dopamine Agonist dosage (rotigotine/Neupro):    4c. Dose agoniste dopaminergique (rotigotine/Neupro):', '4d. Dopamine Agonist dosage (apomorphine/Movapo):    4d. Dose agoniste dopaminergique (apomorphine/Movapo):', "4e. Name of other Dopamine agonist:    4e. Nom d'autre agoniste dopaminergique:", '4f. Dopamine Agonist dosage (Other):    4f. Dose agoniste dopaminergique (autre):', "5. Since taking a dopamine agonist, has the patient shown:    5. Depuis la prise d'agoniste dopaminergique, le patient a-t-il/elle démontré:", "Please specify other dopamine agonist side effect:    Veuillez préciser si vous avez d'autres effets secondaires liés à la prise d'agonistes dopaminergiques:   ", '6. Current MAO-B Inhibitor Medications: (Monoamine Oxidase)    6. Médication actuelle MAO-B Inhibiteur: (Monoamine Oxidase)  ', '6a. MAO-B Inhibitor dosage (rasagiline/Azilect):    6a. MAO-B Inhibiteur dose (rasagiline/Azilect):', '6b. MAO-B Inhibitor dosage (selegiline/Eldepryl):    6b. MAO-B Inhibiteur dose (selegiline/Eldepryl):', '6c. MAO-B Inhibitor dosage (safinamide/Onstryv):    6c. Dose inhibiteur MAO-B (safinamide/Onstryv):', "6d. Name of other MAO-B Inhibitor:    6d. Nom d'un autre inhbibiteur  MAO-B:", '6e. MAO-B Inhibitor dosage (Other):    6e. Dose inhibiteur MAO-B (autre):', '7. Other PD medications:    7. Autres médicaments anti-parkinsoniens:', '7a. Dosage (amantadine/Symmetrel):    7a. Dose (amantadine/Symmetrel):', "7b. Dosage anticholinergics (biperiden, orphenadrine, procyclidine, benztropine):    7b. Dose  d'anticholinergiques (biperiden, orphenadrine, procyclidine, benztropine):", '7c. If other precise medication name and dosage (if known):    7c. Si autre médicament précis, nom et dosage (si connu) :', "8. Total LED (if participant is not on any medications to treat the symptoms of Parkinson's disease, mark as 0 [zero]: ", '9. Non-PD Medications:    9. Médicaments non-PD:', 'Please specify name of other medication:    Veuillez préciser le nom de votre autre médication : ', '9a. Benzodiazepines prescribed for:    9a. Benzodiazépines prescrits pour:', '10. Has the patient ever had a psychiatric disorder?    10. Le patient a-t-il/elle déjà eu un trouble psychiatrique?', 'Please specify which other psychiatric disorder:    Veuillez préciser si autre trouble psychiatrique:', '11. Is the participant followed by health professionals for mental health?     11. Le participant est-il suivi par des professionnels de la santé pour la santé mentale?', '1. Have you ever been diagnosed with:    1. Avez-vous déjà été diagnostiqué pour:', 'Please specify other:    Veuillez préciser autre:', "2. Have you been admitted to the hospital in the past 18 months?    2. Avez-vous été admis à l'hôpital dans les 18 derniers mois?", "2a. How many times were you admitted to the hospital?    2a. Combien de fois avez-vous été admis à l'hôpital?", "2b. Reason for admission:    2b. Raison de l'admission:", "Please specify other reason for hospital admission:    Veuillez préciser si autre raison pour l'admission à l'hôpital:", '2c. Was the altered mental state due to:    2c. Le changement de santé mentale at-il été causé par:', 'Please specify other reason for altered mental state:    Veuillez préciser si autre raison pour changement de santé mentale:', '3. Have you ever experienced a hard blow to the head with or without loss of consciousness? Examples include: falling from a high place, getting hit in the head during a sport, or getting hit in the head due to an accident.    3. Avez-vous déjà reçu un co', '3a. How many times?    3a. Combien de fois?', '4. Have you ever played high contact sports? (e.g. football, hockey, soccer, rugby)    4. Avez-vous déjà joué à des sports de contact? (exemple : football, hockey, soccer, rugby)  ', "4a. How many years did you play high contact sports?    4a. Combien d'années avez-vous joué aux sports de contact? ", '5. Do you suffer from constipation?    5. Souffrez-vous de constipation?', '5a. How many years have you been suffering from constipation?  ', '6. Do you often (e.g. every day or multiple times (4+) a week) feel light-headed or dizzy?    6. Vous sentez-vous souvent (par exemple tous les jours ou plusieurs fois (4+) par semaine) étourdi?', '7. Do you ever have an irrepressible urge to move your legs?    7. Avez-vous déjà ressenti un besoin irrépressible de bouger vos jambes? ', "8. Do you feel or have people told you that your sense of smell is getting worse?    8. Avez-vous remarqué ou vous a-t-on fait remarquer que vous aviez une perte d'odorat? ", '9. Have your friends or family told you lately that you are becoming more forgetful?    9. Est-ce que vos amis ou votre famille vous ont dit dernièrement que vous oubliez des choses plus fréquemment?', '10. Have you noticed problems with your short term memory?    10. Avez-vous remarqué que vous aviez des problèmes de mémoire à court terme?   ', '11. Do you have trouble falling asleep?    11. Avez-vous des difficultés pour vous endormir? ', "11a. Since when have you had trouble falling asleep? (How many years?)    11a. Depuis quand avez-vous des difficultés à vous endormir ? (Combien d'années?)", '12. Do you have trouble staying asleep?    12. Avez-vous des difficultés à rester endormi?', "12a. Since when have you had trouble staying asleep? (How many years?)    12a. Depuis quand avez-vous des difficultés à rester endormi ? (Combien d'années?)? ", "13. What are the chances that you would doze off or fall asleep when you don't intend to? (e.g. when reading, watching TV, driving)    13. Quelle est la probabilité que vous somnoliez ou que vous vous endormiez accidentellement? (exemple : en lisant, en r", "13a. Since when have you had trouble dozing off or falling asleep when you don't intend to? (How many years?)    13a. Depuis quand somnolez-vous ou endormissez-vous accidentellement? (Combien d'années?)", '14. Have you ever been told or have you ever noticed that you were acting out your dreams?(e.g. punching the air, whipping the air, running movements)    14. Vous a-t-on déjà dit ou avez-vous déjà remarqué que vous reproduisiez physiquement la gestuelle d', "14a. Since when have you been acting out your dreams? (How many years?)    14a. Depuis quand cela se produit-il (Combien d'années)? ", '15. In the last month, have you experienced any unexplained pain (that was not due to known conditions e.g. arthritis)?    15. Au cours du dernier mois, avez-vous eu des douleurs inexpliquées (qui ne sont pas causées par une condition connue, exemple : ar', '16. In which environment have you mostly lived?    16. Dans quel environnement avez-vous vécu de manière prépondérante?', '16a.', '16b.', '17. In which environment have you mostly worked?    17. Dans quel environnement avez-vous travaillé de manière prépondérante? ', '17a. ', '17b. ', '18. Have you ever used or been exposed to pesticides? (Insecticides, herbicides, fungicides, rodenticides)    18. Avez-vous déjà utilisé ou avez-vous déjà été exposé à des pesticides? (Insecticides, herbicides, fongicides, rongicides)', '19. Have you done any welding on a regular basis or had any exposure to welding materials?    19. Avez-vous déjà fait de la soudure régulièrement ou avez-vous déjà été exposé à des produits de soudure?', "20. Do you exercise on a regular basis?    20. Faites-vous de l'exercice régulièrement? ", "20a. How often do you exercise per week on average?    20a. Combien de fois faites-vous de l'exercice chaque semaine?", "20b. How long is one exercise session on average?    20b. Quelle est la durée moyenne d'une séance d'exercice? ", "20c. How many times a week do you do high intensity exercise (impossible to have a conversation)?    20c. Combien de fois par semaine faites-vous des exercices d'intensité élevé (impossible de tenir une conversation)? ", "20d. How many times per week do you exercise at moderate intensity (able to carry on a conversation)?    20d. Combien de fois par semaine faites-vous des exercices d'intensité modérée (capable de tenir une conversation)? ", "20e. How many times a week do you exercise at a low intensity (slightly out of breath)?    20e. Combien de fois par semaine faites-vous des exercices d'intensité faible (peu essouflé)? ", "21. What type of exercise do you do?    21. Quel type d'exercice faites-vous?", "Please specify other exercise:    Veuillez préciser autre type d'exercice:  ", '22. Are you currently receiving any of the following therapies?    22. Suivez-vous présentement une ou plusieurs des thérapies suivantes? ', 'Please specify other therapy:    Veuillez préciser quelle autre thérapie:', '22a. Have you had any contact with a PD nurse for care since your last clinic visit?   ', '23. Do you currently drink coffee?    23. Buvez-vous du café régulièrement?', "23a. Since when have you been drinking coffee? (How many years?)    23a. Depuis quand buvez-vous du café? (Combien d'année(s)?) ", '23b. How many cups of coffee per day?    23b. Combien de tasses de café buvez-vous par jour?', '23c. Have you ever drank coffee on a regular basis in the past?    23c. Avez-vous bu du café régulièrement par le passé?', '23d. How many cups of coffee per day did you drink in the past?    23d. Combien de tasses de café buviez-vous par le passé?', "23e. For how many years did you drink coffee?    23e. Pendant combien d'années avez-vous bu du café?", '24. Do you currently use any of the following?    24. Consommez-vous les substances suivantes?', "24a. Since when have you been drinking alcohol? (How many years?)    24a. Depuis quand buvez-vous de l'alcool? (Combien d'année(s)?)", "24b. How many glasses per week?    24b. Combien de verres d'alcool consommez-vous par semaine?", "24c. (If you do NOT currently drink alcohol):  Have you ever drank alcohol on a regular basis in the past?    24c. (Si vous ne buvez pas d'alcool actuellement) :   Consommiez-vous de l'alcool régulièrement par le passé?  ", "24d. For how many years did you drink alcohol?    24d. Pendant combien d'années avez-vous bu de l'alcool?", "24e. How many glasses per week?    24e. Combien de verres d'alcool buvez-vous par semaine?", "24f. Since when have you been smoking? (How many years?)    24f. Depuis quand fumez-vous? (Combien d'année(s)?)", '24g. How many cigarettes per day?    24g. Combien de cigarettes par jour? ', '24h. (If you do NOT currently smoke cigarettes):  Have you ever smoked cigarettes on a regular basis in the past?    24h. (Si lvous ne fumez pas actuellement):  Avez-vous déjà fumé des cigarettes régulièrement par le passé?  ', "24i. For how many years did you smoke?    24i. Pendant combien d'année avez-vous fumé?", '24j. How many cigarettes a day did you smoke?    24j. Combien de cigarettes fumiez-vous par le passé?', '24k. Please specify which recreational drugs:    24k. Veuillez préciser quelles drogues récréatives:', '24l. Please specify what kind of cannabis:    24l. Veuillez préciser quel type de cannabis:', "25. Does/did your biological father have Parkinson's disease?    25. Votre père biologique est / était-il atteint de la maladie Parkinson?", "25a. What is the ethnicity of your biological father?    25a. Quelle est l'ethnicité de votre père biologique?", 'Please specify other ethnicity:    Veuillez préciser autre ethnicité:  ', "26. Does/did your biological mother have Parkinson's disease?    26. Votre mère biologique est/était-elle atteinte de la maladie Parkinson?", "26a. What is the ethnicity of your biological mother?    26a. Quelle est l'ethnicité de votre mère biologique?", 'Please specify other ethnicity:    Veuillez préciser autre ethnicité:', "27. Do you have any brothers or sisters with Parkinson's disease?    27. Avez-vous des frères et des sœurs qui ont la maladie de Parkinson?", '27a. Number of brothers with PD:    27a. Nombre de frères atteints de la maladie de Parkinson: ', '27b. Number of sisters with PD:    27b. Nombre de sœurs atteints de la maladie Parkinson: ', "28. Do you have any children with Parkinson's disease?    28. Avez-vous des enfants atteints de la maladie de Parkinson?", '28a. Number of sons with PD:    28a. Nombre de fils avec la maladie Parkinson:', '28b. Number of daughters with PD:    28b. Nombre de filles atteintes de la maladie de Parkinson:', "29. Do you have grandparents with Parkinson's disease?    29. Avez-vous des grands-parents atteints de la maladie de Parkinson?", '29a. Grandparents with PD:    29a. Grands-parents atteints de la maladie de Parkinson:', "30. Do you have any uncles or aunts with Parkinson's disease?    30. Avez-vous des oncles ou des tantes atteints de la maladie de Parkinson?", '30a. Number of maternal aunts/uncles with PD:    30a. Nombre de tantes/oncles maternels atteints de la maladie de Parkinson:', '30b. Number of paternal aunts/uncles with PD:    30b. Nombre de tantes/oncles paternels atteints de la maladie de Parkinson:', "31. Do you have any cousins with Parkinson's disease?    31. Avez-vous des cousins atteints de la maladie de Parkinson?", '31a. Number of maternal cousins with PD:    31a. Nombre de cousins maternels atteints de la maladie de Parkinson:', '31b. Number of paternal cousins with PD:    31b. Nombre de cousins paternels atteints de la maladie de Parkinson:', "32. Do you have any nephews or nieces with Parkinson's disease?    32. Avez-vous des neveux ou des nièces qui ont la maladie de Parkinson?", '32a. Number of nephews with PD:    32a. Nombre de neveux atteints de la maladie de Parkinson:', '32b. Number of nieces with PD:    32b. Nombre de nièces atteints de la maladie de Parkinson:', "33. Have you ever had an MRI scan for medical reasons or for study purposes?    33. Avez-vous déjà passé un examen par IRM pour des raisons médicales ou pour les besoins d'une étude ?", '34. Does your body contain metallic implants or any other metallic foreign object?    34. Votre organisme contient-t-il des implants métalliques ou tout autre corps étranger métalliques?', "Additional Comments:   (Anything else you'd like to tell us)    Commentaires supplémentaires:  (S'il y a autre chose que vous aimeriez nous dire)    ", 'Was the MoCA administered?     Le MoCA a-t-il été administré?', 'Who administered the MoCA?     Qui a administré le MoCA?', 'How was the MoCA administered?     Comment le MoCA a-t-il été administré?', 'MoCA Test Version Administered    Version du test MoCA administrée  ', 'Please specify what MoCA Test Version was administered if you selected other:     Veuillez spécifier quelle version de test MoCA a été administrée si vous en avez sélectionné autre:', 'What language was the MoCA administered?     Dans quelle langue le MoCA a-t-il été administré?  ', 'Please specify what language the MoCA Test was administered if you selected other:     Veuillez spécifier quelle langue le test MoCA a été administrée si vous en avez sélectionné autre:', "Date of MoCA administration     Date d'administration du MoCA", 'Question for the coordinator: Was the patient ON or OFF during the MoCA?    Question pour le/la coordinateur/trice: Le patient était-il ON ou OFF durant le MoCA?', 'Question for the participant: do you currently feel ON or OFF during the MoCA? (If your medication is working well = ON)    Question pour le/la participant(e): Sentez-vous ON ou OFF durant le MoCA? (Si votre médicament fonctionne bien = ON)', 'Visuospatial/Executive Score    Score Visuospatial/Éxécutif ', 'Naming Score    Score Dénomination', 'Memory Score    Score Mémoire', 'Attention Score    Score Attention', 'Language Score    Score Langage', 'Abstraction Score    Score Abstraction', 'Delayed Recall Score    Score Rappel', 'Orientation Score    Score Orientation', "TOTAL SCORE (make sure to include extra point for 12 years or less of education):    SCORE TOTAL (assurez-vous d'inclure un point supplémentaire pour 12 ans ou moins d'éducation) : ", "Make sure total score and calculation of total score are the same (will not include extra point for 12 years or less of education):     Assurez-vous que le score total et le calcul du score total sont les mêmes (n'inclura pas de point supplémentaire pour ", "Did the participant receive +1 extra point for 12 years or less of education?    Le participant a-t-il reçu +1 point supplémentaire pour 12 ans ou moins d'études?", 'Comments:     Commentaires: ', 'Assessment completed:     Évaluation remplie:  ', 'Assessment completed by:     Évaluation complétée par:', 'How was the MDS-UPDRS administered?   Comment le MDS-UPDRS a-t-il été administré?', "Is it UPDRS v1.2 (part3) legacy ?  S'agit il d'un héritage de UPDRS v1.2 (partie 3) ?", 'Primary source of information:', '1.1 COGNITIVE IMPAIRMENT', 'Updrs_1_1 value', '1.2 HALLUCINATIONS AND PSYCHOSIS', 'Updrs_1_2 value', '1.3 DEPRESSED MOOD', 'Updrs_1_3', '1.4 ANXIOUS MOOD', 'Updrs_1_4 value', '1.5 APATHY', 'Updrs_1_5 value', '1.6 FEATURES OF DOPAMINE DYSREGULATION SYNDROME', 'Updrs_1_6 value', 'Who is filling out this questionnaire (check the best answer):', '1.7 SLEEP PROBLEMS/PROBLEMES DE SOMMEIL  Over the past week, have you had trouble going to sleep at night or staying asleep through the night? Consider how rested you felt after waking up in the morning.    Au cours de la semaine précédente, avez-vous ren', '1.8 DAYTIME SLEEPINESS/SOMNOLENCE DIURNE  Over the past week, have you had trouble staying awake during the daytime?    Au cours de la semaine précédente, avez-vous rencontré des problèmes pour rester éveillé(e) pendant la journée ?', '1.9 PAIN AND OTHER SENSATIONS/DOULEUR ET AUTRES SENSATIONS  Over the past week, have you had uncomfortable feelings in your body like pain, aches tingling or cramps?    Au cours de la semaine précédente, avez-vous eu des sensations inconfortables dans vot', '1.10 URINARY PROBLEMS/PROBLEMES URINAIRES  Over the past week, have you had trouble with urine control? For example, an urgent need to urinate, a need to urinate too often, or urine accidents?    Au cours de la semaine précédente, avez-vous eu des difficu', '1.11 CONSTIPATION PROBLEMS/PROBLEMES DE CONSTIPATION  Over the past week have you had constipation troubles that cause you difficulty moving your bowels?    Au cours de la semaine précédente, avez-vous eu des problèmes de constipation entrainant des  diff', "1.12 LIGHT HEADEDNESS ON STANDING/SENSATION DE TETE VIDE AU LEVER  Over the past week, have you felt faint, dizzy or foggy when you stand up after sitting or lying down?    Au cours de la semaine précédente, avez-vous eu une sensation d'évanouissement, de", "1.13 FATIGUE  Over the past week, have you usually felt fatigued? This feeling is not part of being sleepy or sad.    Au cours de la semaine précédente, vous êtes vous senti(e) habituellement fatigué(e) ? Ce sentiment  ne fait pas partie du fait d'avoir s", '2.1 SPEECH  Over the past week, have you had problems with your speech?', '2.2 SALIVA & DROOLING  Over the past week, have you usually had too much saliva during when you are awake or when you sleep?', '2.3 CHEWING AND SWALLOWING  Over the past week, have you usually had problems swallowing pills or eating meals? Do you need your pills cut or crushed or your meals to be made soft, chopped or blended to avoid choking?', '2.4 EATING TASKS  Over the past week, have you usually had troubles handling your food and using eating utensils? For example, do you have trouble handling finger foods or using forks, knifes, spoons, chopsticks?', '2.5 DRESSING  Over the past week, have you usually had problems dressing? For example, are you slow or do you need help with buttoning, using zippers, putting on or taking off your clothes or jewelry?', '2.6 HYGIENE  Over the past week, have you usually been slow or do you need help with washing, bathing, shaving, brushing teeth, combing your hair or with other personal hygiene?', '2.7 HANDWRITING  Over the past week, have people usually had trouble reading your handwriting?', '2.8 DOING HOBBIES AND OTHER ACTIVITIES  Over the past week, have you usually had trouble doing your hobbies or other things that you like to do?', '2.9 TURNING IN BED  Over the past week, do you usually have trouble turning over in bed?', '2.10 TREMOR  Over the past week, have you usually had shaking or tremor?', '2.11 GETTING OUT OF BED, A CAR, OR A DEEP CHAIR  Over the past week, have you usually had trouble getting out of bed, a car seat, or a deep chair?', '2.12 WALKING AND BALANCE  Over the past week, have you usually had problems with balance and walking?', '2.13 FREEZING  Over the past week, on your usual day when walking, do you suddenly stop or freeze as if your feet are stuck to the floor.', "3a Is the patient on medication for treating the symptoms of Parkinson's Disease?", "3b If the patient is receiving medication for treating the symptoms of Parkinson's Disease,  mark the patient's clinical state using the following definitions:", '3c Is the patient on Levodopa ?', "3.C1 Time of assessment:   Temps de l'évaluation:", '3.C1 Time last PD medications taken:   Heure de la dernière prise de médicaments pour PD:', '3.C1 Minutes since last levodopa dose:  Minutes depuis la dernière dose de lévodopa:', '3.1 SPEECH  ', 'Updrs_3_1 value', '3.2 FACIAL EXPRESSION  ', 'Updrs_3_2 value', '3.3 RIGIDITY- NECK  ', 'Updrs_3_3_neck value', '3.3 RIGIDITY- RUE  ', 'Updrs_3_3_rue value', '3.3 RIGIDITY- LUE  ', 'Updrs_3_3_lue value', '3.3 RIGIDITY- RLE  ', 'Updrs_3_3_rle value', '3.3 RIGIDITY- LLE  ', 'Updrs_3_3_lle value', '3.4 FINGER TAPPING- R  ', 'Updrs_3_4_r value', '3.4 FINGER TAPPING- L  ', 'Updrs_3_4_l value', '3.5 HAND MOVEMENTS- R  ', 'Updrs_3_5_r value', '3.5 HAND MOVEMENTS- L  ', 'Updrs_3_5_l value', '3.6 PRONATION-SUPINATION MOVEMENTS OF HANDS - R  ', 'Updrs_3_6_r value', '3.6 PRONATION-SUPINATION MOVEMENTS OF HANDS- L  ', 'Updrs_3_6_l value', '3.7 TOE TAPPING - R  ', 'Updrs_3_7_r value', '3.7 TOE TAPPING - L  ', 'Updrs_3_7_l value', '3.8 LEG AGILITY - R  ', 'Updrs_3_8_r value', '3.8 LEG AGILITY - L  ', 'Updrs_3_8_l value', '3.9 ARISING FROM CHAIR  ', 'Updrs_3_9 value', '3.10 GAIT  ', 'Updrs_3_10 value', '3.11 FREEZING OF GAIT  ', 'Updrs_3_11 value', '3.12 POSTURAL STABILITY  ', 'Updrs_3_12 value', '3.13 POSTURE  ', 'Updrs_3_13 value', '3.14 GLOBAL SPONTANEITY OF MOVEMENT (BODY BRADYKINESIA)  ', 'Updrs_3_14', '3.15 POSTURAL TREMOR OF THE HANDS - R  ', 'Updrs_3_15_r value', '3.15 POSTURAL TREMOR OF THE HANDS - L  ', 'Updrs_3_15_l value', '3.16 KINETIC TREMOR OF THE HANDS - R  ', 'Updrs_3_16_r value', '3.16 KINETIC TREMOR OF THE HANDS - L  ', 'Updrs_3_16_l value', '3.17 REST TREMOR AMPLITUDE - RUE  ', 'Updrs_3_17_rue value', '3.17 REST TREMOR AMPLITUDE - LUE  ', 'Updrs_3_17_lue value', '3.17 REST TREMOR AMPLITUDE - RLE  ', 'Updrs_3_17_rle value', '3.17 REST TREMOR AMPLITUDE - LLE  ', 'Updrs_3_17_lle value', '3.17 REST TREMOR AMPLITUDE - Lip/Jaw  ', 'Updrs_3_17_lipjaw value', '3.18 CONSTANCY OF REST TREMOR  ', 'Updrs_3_18 value', 'A. Were dyskinesias (chorea or dystonia) present during examination?', 'B. If yes, did these movements interfere with your ratings?', 'Hoehn and Yahr Stage: ', '4.1 TIME SPENT WITH DYSKINESIAS', 'Updrs_4_1 value', '4.2 FUNCTIONAL IMPACT OF DYSKINESIAS', 'Updrs_4_2 value', '4.3 TIME SPENT IN THE OFF STATE', 'Updrs_4_3 value', '4.4 FUNCTIONAL IMPACT OF FLUCTUATIONS', 'Updrs_4_4 value', '4.5 COMPLEXITY OF MOTOR FLUCTUATIONS', 'Updrs_4_5 value', '4.6 PAINFUL OFF-STATE DYSTONIA', 'Updrs_4_6 value', 'Part I: Non-Motor Aspects of Experiences of Daily Living (nM-EDL)', 'Part II: Motor Aspects of Experiences of Daily Living (M-EDL)', 'Part III: Motor Examination', 'Part IV: Motor Complications', 'Who filled out the questionnaire?    Qui a rempli le questionnaire ?', 'How is the questionnaire completed?     Comment le questionnaire est-il rempli ?', '1. Has the person lost interest in friends, family, or home activities?    1. La personne a-t-elle perdu intérêt pour ses amis, sa famille ou ses activités habituelles ?', 'Severity:    Sévérité:', "2. Does the person lack curiosity in topics that would usually have attracted their interest?    2. La personne manque-t-elle de curiosité pour des sujets qui normalement l'intéressent ?", 'Severity:    Sévérité:2', '3. Has the person become less spontaneous and active - for example, is she/he less likely to initiate or maintain conversation?    3. La personne est-elle devenue moins spontanée et active - par exemple, est-elle moins sujette à initier ou maintenir une c', 'Severity:    Sévérité:3', '4. Has the person lost the motivation to act on their obligations or interests?    4. La personne a-t-elle perdu sa motivation pour agir en lien avec ses obligations ou actions habituelles ?', 'Severity:    Sévérité:4', "5. Is the person less affectionate and/or lacking in emotions when compared to her/his usual self?    5. La personne est-elle moins affectueuse ou manque-t-elle d'émotions par rapport à sa personnalité habituelle ?", 'Severity:    Sévérité:5', '6. Does she/he no longer care about anything?    6. La personne semble-t-elle ne plus se soucier de rien ?', 'Severity:    Sévérité:6', '7. Has the person developed sadness or appear to be in low spirits? Does she/he have episodes of tearfulness?    7. La personne a-t-elle développé de la tristesse ou apparait-elle démoralisée ? A-t-elle des épisodes de pleurs ?', 'Severity:    Sévérité:7', '8. Has the person become less able to experience pleasure?    8. La personne a-t-elle moins de capacité à ressentir du plaisir ?', 'Severity:    Sévérité:8', "9. Has the person become discouraged about their future or feel that she/he is a failure?    9. La personne semble t'elle découragée par rapport à son futur ou a-t-elle l'impression d'être un échec ?", 'Severity:    Sévérité:9', '10. Does the person view herself/himself as a burden to family?    10. La personne se voit-elle comme un fardeau pour son entourage ?', 'Severity:    Sévérité:10', "11. Has the person become unduly anxious or worried about things that are routine (e.g. events, visits, etc.)?    11. La personne est-elle devenue plus anxieuse ou s'inquiète-t-elle de choses faisant partie de la routine (évènements, visites, etc.) ?", 'Severity:    Sévérité:11', '12. Does the person feel very tense, having developed an inability to relax, shakiness or symptoms of panic?    12. La personne se sent-elle très tendue, se sent incapable de se relaxer, a des tremblements ou des symptômes de panique ?', 'Severity:    Sévérité:12', '13. Has the person become agitated, aggressive, irritable, or temperamental?    13. La personne est-elle devenue agitée, agressive, irritable ou caractérielle ?', 'Severity:    Sévérité:13', '14. Has she/he become unreasonably or uncharacteristically argumentative?    14. La personne argumente-t-elle de façon déraisonnable ou inhabituelle ?', 'Severity:    Sévérité:14', '15. Has the person become more impulsive, seeming to act without considering things?    15. La personne est-elle devenue plus impulsive, semblant agir sans réfléchir au préalable ?', 'Severity:    Sévérité:15', '16. Does the person display sexually disinhibited or intrusive behaviour, such as touching (themselves/others), hugging, groping, etc., in a manner that is out of character or may cause offence?    16. La personne présente-t-elle des comportements sexuell', 'Severity:    Sévérité:16', '17. Has the person become more easily frustrated or impatient? Does she/he have troubles coping with delays, or waiting for events or for their turn?    17. La personne est-elle devenue plus facilement frustrée ou impatiente ? A-t-elle de la difficulté à ', 'Severity:    Sévérité:17', "18. Does the person display a new recklessness or lack of judgement when driving (e.g. speeding, erratic swerving, abrupt lane changes, etc.)?    18. La personne est-elle nouvellement intrépide ou présente un manque de jugement lorsqu'elle conduit l'autom", 'Severity:    Sévérité:18', '19. Has the person become more stubborn or rigid, i.e., uncharacteristically insistent on having their way, or unwilling/unable to see/hear other views?    19. La personne est-elle devenue têtue ou rigide i.e. insistante de façon inhabituelle sur ses opin', 'Severity:    Sévérité:19', '20. Is there a change in eating behaviors (e.g., overeating, cramming the mouth, insistent on eating only specific foods, or eating the food in exactly the same order)?    20. La personne présente-t-elle un changement dans ses habitudes alimentaires (ex: ', 'Severity:    Sévérité:20', '21. Does the person no longer find food tasteful or enjoyable? Are they eating less?    21. La personne a-t-elle perdu le plaisir de manger ou de goûter les aliments ? Mange-t-elle moins ?', 'Severity:    Sévérité:21', '22. Does the person hoard objects when she/he did not do so before?    22. La personne accumule-t-elle de façon nouvelle certains objets ?', 'Severity:    Sévérité:22', '23. Has the person developed simple repetitive behaviors or compulsions?    23. La personne a-t-elle développé des compulsions ou des comportements répétitifs ?', 'Severity:    Sévérité:23', '24. Has the person recently developed trouble regulating smoking, alcohol, drug intake or gambling, or started shoplifting?    24. La personne a-t-elle développé une difficulté à contrôler sa consommation de cigarettes, alcool, drogues ou a développé du j', 'Severity:    Sévérité:24', "25. Has the person become less concerned about how her/his words or actions affect others? Has she/he become insensitive to others' feelings?    25. La personne est-elle moins préoccupée par l'effet de ses paroles ou actions sur les autres ? Est-elle deve", 'Severity:    Sévérité:25', '26. Has the person started talking openly about very personal or private matters not usually discussed in public?    26. La personne parle-t-elle ouvertement de ses problèmes personnels ou de sujets privés qui ne se discutent généralement pas en public ?', 'Severity:    Sévérité:26', "27. Does the person say rude or crude things or make lewd sexual remarks that she/he would not have said before?    27. La personne fait-elle des commentaires grossiers ou désobligeants ou fait des remarques sexuelles qu'elle n'aurait pas faites auparavan", 'Severity:    Sévérité:27', "28. Does the person seem to lack the social judgement they previously had about what to say or how to behave in public or private?    28. La personne a-t-elle perdu son jugement social antérieur par rapport à ce qui peut se dire ou la façon d'agir en publ", 'Severity:    Sévérité:28', "29. Does the person now talk to strangers as if familiar, or intrude on their activities?    29. La personne s'adresse-t-elle maintenant à des étrangers comme s'ils étaient familiers ou s'immisce dans leurs activités ?", 'Severity:    Sévérité:29', "30. Has the person developed beliefs that they are in danger or that others are planning to harm them or steal their belongings?    30. La personne a-t-elle développé des croyances à l'effet qu'elle est en danger, ou que les autres complotent pour lui fai", 'Severity:    Sévérité:30', "31. Has the person developed suspiciousness about the intentions or motives of other people?    31. La personne est-elle devenue suspicieuse des intentions ou des motifs d'autrui ?", 'Severity:    Sévérité:31', '32. Does she/he have unrealistic beliefs about their power, wealth or skills?    32. La personne a-t-elle des croyances irréalistes par rapport à ses pouvoirs, richesses ou talents ?', 'Severity:    Sévérité:32', "33. Does the person describe hearing voices or talking to imaginary people or 'spirits'?    33. La personne décrit-elle entendre des voix ou s'entretient-elle avec des personnes imaginaires ou des > ?", 'Severity:    Sévérité:33', '34. Does the person report or complain about, or act as if seeing things (e.g. people, animals or insects) that are not there, i.e., that are imaginary to others?    34. La personne rapporte-elle, se plaint-elle ou agit-elle comme si elle voyait des chose', 'Severity:    Sévérité:34', 'Motivation/Drive (sum of Question 1-6)', 'Mood/Anxiety (sum of Question 7-12)', 'Impulsivity/Delay Gratification (sum of Question 13-24)', 'Societal Norms (sum of Question 25-29)', 'Beliefs/Sensory Experiences (sum of Question 30-34)', 'MBI-C Grand Total', 'Tests completed:     Tests complétés:  ', 'How were the test completed?     Comment les tests ont-ils été remplis?', '1. Sadness/Tristesse', '2. Pessimism/Pessimisme', '3. Past Failure/Échecs dans le passé', '4. Loss of Pleasure/Perte de plaisir', '5. Guilty Feelings/Sentiments de culpabilité', "6. Punishment Feelings/Sentiment d'être puni(e)", '7. Self-Dislike/Sentiments négatifs envers soi-même', '8. Self-Criticalness/Attitude critique envers soi', '9. Suicidal Thoughts or Wishes/Pensées ou désirs de suicide', '10. Crying/Pleurs', '11. Agitation', "12. Loss of Interest/Perte d'intérêt", '13. Indecisiveness/Indécision', '14. Worthlessness/Dévalorisation', "15. Loss of Energy/Perte d'énergie", '16. Changes in Sleeping Pattern/Modifications dans les habitudes de sommeil', '17. Irritability/Irritabilité', "18. Changes in Appetite/Modifications de l'appétit", '19. Concentration Difficulty/Difficulté à se concentrer', '20. Tiredness or Fatigue', "21. Loss of interest in Sex/Perte d'intérêt pour le sexe", 'BDI-II Total Score', "Classification Instructions:  If score = 0-13, select 'Minimal'  If score = 14-19, select 'Mild'  If score = 20-28, select 'Moderate'  If score = 29-63, select 'Severe'", "1. Numbness or tingling/sensations d'engourdissement ou de picotement", '2. Feeling hot/bouffées de chaleur', "3. Wobbliness in legs/'jambes molles', tremblements dans les jambes", '4. Unable to relax/incapacité de se détendre', '5. Fear of the worst happening/crainte que le pire ne survienne', '6. Dizzy or lightheaded/étourdissement ou vertige, désorientation', '7. Heart pounding or racing/battements cardiaques marqués', "8. Unsteady/mal assuré(e), manque d'assurance dans mes mouvements", '9. Terrified/terrifié(e)', '10. Nervous/nervosité', "11. Feeling of choking/sensation d'étouffement", '12. Hands trembling/tremblement de mains', '13. Shaky/tremblements, chancelant(e)', '14. Fear of losing control/crainte de perdre le contrôle', '15. Difficulty breathing/respiration difficile', '16. Fear of dying/peur de mourir', "17. Scared/sensation de peur, 'avoir la frousse'", '18. Indigestion or discomfort in abdomen/indigestion ou malaise abdominal', "19. Faint/sensation de défaillance ou d'évanouissement", '20. Face flushed/rougissement du visage', '21. Sweating (not due to heat)/transpiration (non associée à la chaleur)', 'BAI Total Score', "BAI Classification:  If score = 0-7, select 'Minimal'  If score = 8-15, select 'Mild'  If score = 16-25, select 'Moderate'  If score = 26-63, select 'Severe", '1. Writing', '2. Drawing', '3. Throwing', '4. Scissors', '5. Toothbrush', '6. Knife (without a fork)', '7. Spoon', '8. Broom (upper hand)', '9. Striking Match (match hand)', '10. Opening a box (lid hand)', '11. Which foot do you prefer to kick with?', '12. Which eye do you use when using only one?', 'EHI Total Score Right  ', 'EHI Total Score Left  ', 'EHI Handedness Score  ', "Handedness Classification:   If score < -0.4, select 'Left Handed'  If score -0.4 < = HS < = +0.4, select 'Ambidextrous'  If score > +0.4, select 'Right Handed'  ", 'Questionnaire completed:     Questionnaire rempli:', '1. My motivation is lower when I am fatigued. (Disagree=1, Agree=7)', '2. Exercise brings on my fatigue. (Disagree=1, Agree=7)', '3. I am easily fatigued. (Disagree=1, Agree=7)', '4. Fatigue interferes with my physical functioning. (Disagree=1, Agree=7)', '5. Fatigue causes frequent problems for me. (Disagree=1, Agree=7)', '6. My fatigue prevents sustained physical functioning. (Disagree=1, Agree=7)', '7. Fatigue interferes with carrying out certain duties and responsibilities. (Disagree=1, Agree=7)', '8. Fatigue is among my three most disabling symptoms. (Disagree=1, Agree=7)', '9. Fatigue interferes with my work, family, or social life. (Disagree=1, Agree=7)', 'Total of all questions', 'Divided by 9', 'Unnamed: 0', 'A.1. Feeling anxious or nervous', 'A.2. Feeling Tense or stressed', 'A.3. Being unable to relax ', 'A.4. Excessive worrying about everyday matters', 'A.5. Fear of something bad, or even the worst, happening', 'B.1. Panic or intense fear', 'B.2. Shortness of breath', 'B.3. Heart palpitations or heart beating fast (not related to physical effort or activity)', 'B.4. Fear of losing control', 'C.1. Social situations (where one may be observed, or evaluated by others, such as speaking in public, or talking to unknown people)', 'C.2. Public Setting (situations from which it may be difficult or embarrassing to escape, such as queues or lines, crowds, bridges, or public transportation) ', 'C.3. Specific objects or situations (such as flying, heights, spiders or other animals, needles, or blood) ', 'PAS Score', '1. Had difficulty getting around in public?', '2. Had difficulty dressing yourself?', '3. Felt depressed?', '4. Had problems with your close personal relationships?', '5. Had problems with your concentration, e.g. when reading or watching TV?', '6. Felt unable to communicate with people properly?', '7. Had painful muscle cramps or spasms?', "8. Felt embarrassed in public due to having Parkinson's disease?", 'PDQ-8 Raw Score', 'PDQ-8 Summary Index', '1. Had difficulty doing the leisure activities which you would like to do?/Avez-\xadvous eu des difficultés dans la pratique de vos loisirs?', '2. Had difficulty looking after your home, e.g. DIY, housework, cooking?/Avez-\xadvous eu des difficultés à vous occuper de votre maison, par exemple bricolage, ménage, cuisine?', '3. Had difficulty carrying bags of shopping?/Avez-\xadvous eu des difficultés à porter des sacs de provisions ?   ', '4. Had problems walking half a mile?/Avez\xad�?vous eu des problèmes  pour faire 1 kilomètre à pied ?', '5. Had problems walking 100 yards?/Avez-\xadvous eu des problèmes pour faire 100 mètres à pied ? ', "6. Had problems getting around the house as easily as you would like?/Avez-\xadvous eu des problèmes  à vous déplacer chez vous, aussi aisément que vous l'auriez souhaité ?  ", '7 Had difficulty getting around in public?/Avez-\xadvous eu des difficultés à vous déplacer dans les lieux publics? ', "8. Needed someone else to accompany you when you went out?/Avez-\xadvous eu besoin de quelqu'un pour vous accompagner lors de vos sorties? ", "9. Felt frightened or worried about falling over in  public?/Avez-\xadvous eu peur ou vous êtes-\xadvous senti(e) inquiet(e) à l'idée de tomber en public?", "10. Been confined to the house more than you  would like?/Avez-\xadvous été confine(e) chez vous plus que vous ne l'auriez souhaité? ", '11. Had difficulty washing  yourself?/Avez-\xadvous eu des difficultés   pour vous laver?', '12. Had difficulty dressing  yourself?/Avez\xad�?vous eu des difficultés pour vous habiller?', '13. Had problems doing up your shoe laces?/Avez-\xad�?vous eu des problèmes pour  boutonner vos vêtements ou pour lacer vos chaussures?', '14. Had problems writing clearly?/Avez\xad�?vous eu des problèmes pour écrire lisiblement?', '15. Had difficulty cutting up your food?/Avez-\xadvous eu des difficultés pour couper la nourriture?', '16. Had difficulty holding a drink without spilling it?/Avez-vous eu des difficultés pour tenir un verre sans le renverser?  ', '17. Felt depressed?/Vous êtes-\xad�?vous senti(e) déprimé(e)?', '18. Felt isolated and lonely?/Vous êtes�?vous senti(e) isolé(e) et seul(e)?', '19. Felt weepy or tearful?/Vous êtes-\xadvous senti(e) au  bord des larmes ou avez-\xadvous pleuré?', "20. Felt angry or bitter?/Avez-\xadvous ressenti de la colère ou de l'amertume ?", '21. Felt anxious?/Vous êtes-\xadvous senti(e)  anxieux(se) ?  ', '22. Felt worried about your future?/Vous êtes-\xad�?vous senti(e) inquiet(e) pour votre avenir?', "23. Felt you had to conceal your Parkinson's from people?/Avez-\xadvous ressenti le besoin de dissimuler aux autres  votre maladie?", '24. Avoided situations which involve eating or drinking in public?/Avez-\xadvous évité des situations où vous deviez manger ou boire en public?  ', "25. Felt embarrassed in public due to having Parkinson's disease?/Vous êtes-\xadvous senti(e) gêné(e) en public à cause de votre maladie de Parkinson?", "26. Felt worried by other people's reaction to you?/Vous êtes-\xad�?vous senti(e) inquiet(e) des réactions  des autres à votre égard?", '27. Had problems with your close personal relationships?/Avez-\xadvous eu des problèmes dans vos relations avec vos proches ?  ', "28. Lacked support in the ways you need from your spouse or partner?/Avez-\xadvous manqué du soutien,  dont  vous  aviez besoin, de la part de votre époux(se) ou conjoint(e)? (If you do not have a spouse or partner tick never/Si vous n'avez pas de conjoint o", '29. Lacked support in the ways you need from your family or close friends?/Avez-\xad�?vous manqué du soutien dont vous aviez besoin,  de la part de votre famille ou de vos amis proches?', '30. Unexpectedly fallen asleep during the day?/Vous êtes-\xad�?vous endormi(e) dans la journée de façon inattendue?  ', '31. Had problems with your concentration, e.g. when reading or watching TV?/Avez�?vous eu des problèmes de  concentration, par  exemple en lisant ou en regardant la télévision?', '32. Felt your memory was bad?/Avez�?vous senti que votre mémoire était mauvaise?', '33.  Had distressing dreams or hallucinations?/Avez\xad�?vous fait des mauvais rêves ou eu des hallucinations?', '34 Had difficulty with your speech?/Avez-\xadvous eu des difficultés pour parler?  ', '35. Felt unable to communicate with people properly?/Vous êtes-\xadvous senti(e) incapable de communiquer normalement avec les autres?', '36. Felt ignored by people?/Vous êtes\xad�?vous senti(e) ignoré(e) par les autres?  ', '37. Had painful muscle cramps or spasms?/Avez vous eu des crampes ou des spasmes musculaires douloureux?', '38. Had aches and pains in your joints or body?/Avez-\xadvous eu mal ou avez-vous eu des douleurs dans les articulations ou dans le corps?', '39. Felt unpleasantly hot or cold?/Avez-\xadvous eu la sensation désagréable de chaud ou de froid?', 'PDQ-39 Mobility (Raw Score)', 'PDQ-39 Mobility (Scale Score)', 'PDQ-39 Activities of Daily Living  (Raw Score)', 'PDQ-39 Activities of Daily Living (Scale Score)', 'PDQ-39 Emotional Well Being (Raw Score)', 'PDQ-39 Emotional Well Being (Scale Score)', 'PDQ-39 Stigma (Raw Score)', 'PDQ-39 Stigma (Scale Score)', 'PDQ-39 Social Support (Raw Score)', 'PDQ-39 Social Support (Scale Score)', 'PDQ-39 Cognition (Raw Score)', 'PDQ-39 Cognition (Scale Score)', 'PDQ-39 Communication (Raw Score)', 'PDQ-39 Communication (Scale Score)', 'PDQ-39 Bodily Discomfort (Raw Score)', 'PDQ-39 Bodily Discomfort (Scale Score)', 'PDQ-39 Total Raw Score ', 'PDQ-39 Summary Index', 'Please carefully read each item in the list below. Mark one box based on how closely you agree with the statement below.', 'Score', '1. In the past month, have you had difficulty swallowing or have you choked?', '2. In the past month, has saliva dribbled out of your mouth?', '3. In the past month, has food ever become stuck in your throath?', '4. In the past month, did you ever have the feeling during a meal that you were full very quickly?', '5. Constipation is a blockage of the bowel, a condition in which someone has a bowel movement twice a week or less.  In the past month, have you had problems with constipation?', '6. In the past month, did you have to strain hard to pass stools?', '7. In the past month, have you had involuntary loss of stools?', "Questions 8 to 13 deal with problems with passing urine. If you use a catheter you can indicate this by placing a cross in the box 'use cathether'.", '8. In the past month, have you had difficulty retaining urine?', '9. In the past month, have you had involuntary loss of urine?', '10. In the past month, have you had the feeling that after passing urine your bladder was not completely empty?', '11. In the past month, has the stream of urine been weak?', '12. In the past month, have you had to pass urine again within 2 hours of the previous time?', '13. In the past month, have you had to pass urine at night?', '14. In the past month, when standing up have you had the feeling of either becoming lightheaded, or no longer being able to see properly, or no longer being able to think clearly?', '15. In the past month, did you become light-headed after standing for some time?', '16. Have you fainted in the past 6 months?', '17. In the past month, have you ever perspired excessively during the day?', '18. In the past month, have you ever perspired excessively during the night?', '19. In the past month, have your eyes ever been over-sensitive to bright light?', '20. In the past month, how often have you had trouble tolerating cold?', '21. In the past month, how often have you had trouble tolerating heat?', 'What is the gender of the participant?', '22. In the past month, have you been impotent (unable to have or maintain an erection)?', '23. In the past month, how often have you been unable to ejaculate?', '23a. In the past month, have you taken medication for an erection disorder? ', 'Which medication have you taken for an erection disorder?', '24. In the past month, was your vagina too dry during sexual activity?', '25. In the past month, have you had difficulty reaching an orgasm?', 'a. constipation?', 'What medication have used?', 'd. urinary problems?', 'What medication have used?2', 'e. blood pressure?', 'What medication have used?3', "f. other symptoms (not symptoms related to Parkinson's disease)", 'What medication have used?4', 'Gastrointestinal dysfunction ', 'Urinary dysfunction ', 'Cardiovascular dysfunction', 'Thermoregulatory dysfunction ', 'Pupillomotor dysfunction ', 'Sexual dysfunction men ', 'Sexual dysfunction women', 'Total Score', '1. Are you interested in learning new things?', '2. Does anything interest you?', '3. Are you concerned about your condition?', '4. Do you put much effort into things?', '5. Are you always looking for something to do?', '6. Do you have plans and goals for the future?', '7. Do you have motivation?', '8. Do you have the energy for daily activities?', '9. Does someone have to tell you what to do each day?', '10. Are you indifferent to things?', '11. Are you unconcerned with many things?', '12. Do you need a push to get started on things?', '13. Are you neither happy nor sad, just in between?', '14. Would you consider yourself apathetic?', 'Apathy Scale Score', 'Neuropsycholgical Test Date:', 'Age at Neuropsych:', "How is the assessment completed?     Comment l'évaluation est-il rempli?", 'Was the CDT test administered?', 'Command Clock raw (max 10)', 'Command Clock z score', 'Copy Clock raw (max 10)', 'Copy Clock z score', 'Was the Brixton test administered?', 'Brixton raw score', 'Brixton z score', 'Was the Stroop Colour and Word test administered?', 'Colour Words raw:', 'Colour words t score ', 'Colour Words z score', 'Was the Trails A B test administered?', 'Trail A raw score', 'Trail A Errors raw score ', 'Trail A t score', 'Trail A z score', 'Trail B raw score', 'Trail B Errors raw score', 'Trail B t score', 'Trail B z score', 'Was the WAIS IV Digit Span test administered?', 'Digit Span Forward raw', 'Digit Span Forward scaled scores', 'Digit Span Forward z score', 'Digit Span Backward raw', 'Digit span Backward scaled score', 'Digit Span Backward z score ', 'Digit Span Sequencing raw', 'Digit Span Sequencing scaled score ', 'Digit Span Sequencing z score ', 'Digit Span Total raw', 'Digit Span total scaled score ', 'Digit Span Total z score', 'Was the Letter Fluency test administered?', 'FAS total raw', 'FAS Repetition raw ', 'FAS Rule Break raw', 'FAS t score', 'FAS z score ', 'Was the Semantic Fluency test administered?', 'Animals total raw', 'Animals Repetition raw', 'Animals Rule Break raw', 'Animals t score', 'Animals z score ', 'Actions total raw ', 'Actions t score', 'Actions z score', 'Actions Rule Break raw', 'Actions Repetition raw', 'Actions Repetition t score ', 'Actions Repetition z score', 'Was the Boston Naming test administered?', 'Total (SR+SC) raw', 'SR+SC+PC raw', 'Semantic Cues', 'Phonemic Cues', 'Total (SR+SC) t score', 'Total (SR+SC) z score', 'Was the Rey Complex Figure Test administered?', 'Copy raw', 'Copy z score', 'Copy % range:If score = 1, select: > 16If score = 2, select: 11-16If score = 3, select: 6-10If score = 4, select: 2-5if score = 5, select: < = 1', 'Time raw', 'Time % range:If score = 1, select: > 16If score = 2, select: 11-16If score = 3, select: 6-10If score = 4, select: 2-5if score = 5, select: < = 1', 'Immediate Recall raw', 'Immediate Recall t score', 'Immediate Recall z score', 'Delayed Recall raw', 'Delayed Recall t score', 'Delayed Recall z score', 'Recog raw', 'Recog t score', 'Recog z score', 'Was the Hopkins Verbal Learning Test Revised administered?', 'Trial 1 raw', 'Trial 2 raw', 'Trial 3 raw', 'Total Recall raw', 'Total Recall t score', 'Total Recall z score', 'Delayed Recall raw2', 'Delayed Recall t score3', 'Delayed Recall z score ', 'Retention (%)', 'Retention t score ', 'Retention z score', 'Recog. Disc. Index Raw', 'Recog. Disc. Index T score', 'Recog. Disc. Index z sore ', 'Memory z score', 'Visio-Perceptual Function z score', 'Attention z score', 'Executive Function z score', 'Language z score', 'Global Cognition z score', 'Cognitive Status', 'Administered by / Administré par : ', "Age at evaluation / Âge au moment de l'évaluation:", 'Native language / Langue maternelle (choice=English / Anglais)', 'Native language / Langue maternelle (choice=French / Français)', 'Native language / Langue maternelle (choice=Other / Autre)', "Years of education /  nombre d'années d'éducation (calculated by neuropsy):  ", 'Other / Autre', "Language of evaluation / Langue d'évaluation:   (choice=English / Anglais)", "Language of evaluation / Langue d'évaluation:   (choice=French / Français)", "Language of evaluation / Langue d'évaluation:   (choice=Other / Autre)", 'Other / Autre2', 'Normal cognition (superior or equal to 26 on MoCA)', 'Self-reported medical history  Histoire médicale auto-rapportée', "Subjective Complaint: more difficulties with memory, judgement, concentration, planning, etc. than before    Plainte subjective: plus de difficulté qu'avant avec la mémoire, le jugement, la concentration, la planification, etc.", 'Functional Impact: the cognitive complaints affect daily activities    Impact fonctionnel: la plainte cognitive affecte les activités du quotidien', 'Details for missing data', 'Trial total 1,2,3 (Raw score)', 'Intrusions total 1,2,3 (Raw score)', 'Repetitions total 1,2,3 (Raw score)', 'Trial 4 delayed (Raw score)', 'Intrusion 4', 'Repetition 4', 'Comments', 'Details for missing data3', 'Copy time (sec)', 'Copy strategy / Stratégie de copie', 'Immediate recall raw', 'Immediate recall time (sec)', 'Delayed recall raw', 'Delayed recall time (sec)', 'Comments4', 'Details for missing data5', 'Digit Span Forward - total correct (Raw score) ', 'Digit span forward - longest correct serie (Raw score)', 'Digit Span Backward - total correct (Raw score)', 'Digit span backward - longest correct serie (Raw score)  ', 'Comments6', 'Details for missing data7', 'Trail A raw score (time in sec.)', 'Trail B raw score (time in sec.)', 'Comments8', 'Details for missing data9', 'Comments10', 'Was the Stroop Colour and Word test (Golden) administered?', 'Details for missing data11', 'STROOP GOLDEN : WORDS, number of responses', 'STROOP GOLDEN : words, self-corrected errors (raw score)', 'STROOP GOLDEN, words, uncorrected errors (raw score)', 'STROOP GOLDEN : COLORS Number of responses', 'STROOP GOLDEN : colors, self-corrected errors (raw scores)', 'STROOP GOLDEN: colors, uncorrected errors', 'Stroop GOLDEN: INK Number of Responses ', 'Stroop GOLDEN : ink, self-corrected errors (raw score)', 'Stroop GOLDEN, ink, uncorrected errors (raw score)', 'Was the Stroop Colour and Word (D-KEFS) test administered?', 'Details for missing data12', 'Stroop - D-Kefs, Cond. 1 - COLORS : Time (sec) (Raw score)', 'Stroop - D-Kefs - cond. 1: self-corrected errors (Raw score)', 'Stroop - D-Kefs - Cond.1: Uncorrected errors (Raw score)', 'Stroop - D-Kefs - Cond.1: Total errors (Raw score)', 'Stroop - D-Kefs - Cond.1: Total errors (Automatic Calculation)', 'Stroop - D-Kefs - Cond.2 WORDS: Time (sec) (Raw score)', 'Stroop - D-Kefs -  Cond.2: Self-corrected errors (Raw score)', 'Stroop - D-Kefs - Cond.2: uncorrected errors (Raw score)', 'Stroop - D-Kefs - Cond.2: total errors (Raw score)', 'Stroop - D-Kefs - Cond.2: total errors (Automatic Calculation)', 'Stroop - D-Kefs - Cond.3 - INK: Time (sec) (Raw score)', 'Stroop - D-Kefs - Cond.3: self-corrected errors (Raw score)', 'Stroop - D-Kefs - Cond.3: Uncorrected errors (Raw score)', 'Stroop - D-Kefs - Cond.3: Total errors (Raw score)', 'Stroop - D-Kefs - Cond.3: Total errors (Automatic calculation)', 'Stroop - D-Kefs - Cond. 3 - 1 (time)', 'Stroop - D-Kefs - Cond. 3 - 1 (time) (Automatic Calculation)', 'Comments13', 'Details for missing data14', 'Letter Fluency F (Raw score)', 'Letter Fluency A (Raw score)', 'Letter Fluency S (Raw score)', 'Letter Fluency Total (Raw score)', 'Total Set-Loss errors', 'Total repetition errors ', 'Letter Fluency Total (Automatic Calculation)', 'Comments15', 'Details for missing data16', 'Comments17', 'Details for missing data18', 'Semantic Fluency Animals (Raw score)', 'Semantic Fluency Actions (Raw score)', 'Semantic Fluency Total (Raw score)', 'Total Set-Loss errors19', 'Total Repetition errors', 'Semantic Fluency Total (Automatic Calculation)', 'Comments20', 'Details for missing data21', 'BNT sans indice (Raw score)', 'BNT sans indice + IS', 'BNT sans indice + IS + IP', 'Comments22', 'Was the Purdue pegboard administered?', 'Details for missing data23', 'Main dominante (30 sec)', 'Main dominante (1 min)', 'Main non dominante (30 sec)', 'Main non dominante (1 min)', 'Deux mains (30 sec)', 'Comments24', 'Comments25', 'Native language / Langue maternelle', "Language of evaluation / Langue d'évaluation:  ", 'Assessment completed:     Évaluation remplie:', 'How was the UPDRS administered?   Comment le UPDRS a-t-il été administré?', '1. Speech', '2. Facial Express', '3a. Tremor/ Rest (face, lips chin)', '3b. Tremor/ Rest Hand (right)', '3c. Tremor/ Rest Hand (left)', '3d. Tremor/ Rest Foot (right)', '3e. Tremor/ Rest Foot (left)', '4a. Action: Hand (right)', '4b. Action: Hand (left)', '5a. Rigidity: Neck', '5b. Rigidity: Upper Extremity (right hand)', '5c. Rigidity: Upper Extremity (left hand)', '5d. Rigidity: Lower Extremity (right leg)', '5e. Rigidity: Lower Extremity (left leg) ', '6a. Finger Taps (right)', '6b. Finger Taps (left)', '7a. Hand Mov. (right)', '7b. Hand Mov. (left)', '8a. Rapid Alt Hand Mov. (right)', '8b. Rapid Alt Hand Mov. (left)', '9a. Leg Agility (right)', '9b. Leg Agility (left)', '10. Arise Chair', '11. Posture', '12. Gait', '13. Postural Stability', '14. Body Bradykinesia & Hypokinesia  ', 'UPDRS Total', 'Tremor Total', 'Rigidity Total', 'Ratio Tremor/ Rigidity Total', 'Right Part Score Total', 'Left Part Score Total', 'Laterality Index (+=right) ', '1. I am interested in things.', '2. I get things done during the day.', '3. Getting things started on my own is important to me. ', '4. I am interested in having new experiences. ', '5. I am interested in learning new things. ', '6. I put little effort into anything.', '7. I approach life with intensity. ', '8. Seeing a job through to the end is important to me. ', '9. I spend time doing things that interest me.', '10. Someone has to tell me what to do each day.', '11. I am less concerned about my problems than I should be. ', '12. I have friends. ', '13. Getting together with friends is important to me.', '14. When something good happens, I get excited. ', '15. I have an accurate understanding of my problems. ', '16. Getting things done during the day is important to me. ', '17. I have initiative. ', '18. I have motivation. ', 'What is your relationship to the person you are filling out the questionnaire for?     Quelle est votre relation avec la personne pour laquelle vous remplissez le questionnaire?', '1. S/he is interested in things', '2. S/he gets things done during the day', '3. Getting things started on his/her own is important to him/her.', '4. S/he is interested in having new experiences', '5. S/he is interested in learning new things', '6. S/he puts little effort into anything', '7. S/he approaches life with intensity', '8. Seeing a job through to the end is important to him/her', '9. S/he spends time doing things that interest him/her', '10. Someone has to tell him/her what to do each day', '11. S/he is less concerned about her/his problems than s/he should be', '12. S/he has friends', '13. Getting together with friends is important to him/her', '14. When something good happens, s/he gets excited', '15. S/he has an accurate understanding of her/his problems', '16. Getting things done during the day is important to her/him', '17. S/he has initiative', '18. S/he has motivation', 'How is the test completed?     Comment le test est-il complété?', 'Le patient a-t-il/elle fait le test:', 'Temps (secondes)', 'Temps (secondes)2', 'Temps (secondes)3', 'Si fluctuations motrices pendant le test, le patient se considère-t-il/elle:', "1. AVANT BLESSURE OU MALADIE, je ne parle que quand on m'adresse la parole", "1. AUJOURD'HUI, je ne parle que quand on m'adresse la parole", "2. AVANT BLESSURE OU MALADIE, je m'agace ou me fâche facilement. Je suis si facilement irritable que je m'emporte sans raison. ", "2. AUJOURD'HUI, je m'agace ou me fâche facilement. Je suis si facilement irritable que je m'emporte sans raison. ", '3. AVANT BLESSURE OU MALADIE, je répète certaines actions ou me braque sur certaines idées. ', "3. AUJOURD'HUI, je répète certaines actions ou me braque sur certaines idées. ", "4. AVANT BLESSURE OU MALADIE, j'agis de façon impulsive. ", "4. AUJOURD'HUI, j'agis de façon impulsive. ", "5. AVANT BLESSURE OU MALADIE, je mélange l'orde des choses, je me sens désorienté(e) quand il faut faire plusieurs choses à la suite. ", "5. AUJOURD'HUI, je mélange l'orde des choses, je me sens désorienté(e) quand il faut faire plusieurs choses à la suite. ", '6. AVANT BLESSURE OU MALADIE, je ris ou je pleure trop facilement. ', "6. AUJOURD'HUI, je ris ou je pleure trop facilement. ", "7. AVANT BLESSURE OU MALADIE, je fais et refais toujours la même erreur, je n'arrive pas à retenir les expériences passées. ", "7. AUJOURD'HUI, je fais et refais toujours la même erreur, je n'arrive pas à retenir les expériences passées. ", "8. AVANT BLESSURE OU MALADIE, j'ai du mal à commencer une activité, je manque d'initiatie, de motivation. ", "8. AUJOURD'HUI, j'ai du mal à commencer une activité, je manque d'initiatie, de motivation. ", "9. AVANT BLESSURE OU MALADIE, je fais des commentairs et des propositions d'ordre sexuel hors de propos, je suis trop dragueur. ", "9. AUJOURD'HUI, je fais des commentairs et des propositions d'ordre sexuel hors de propos, je suis trop dragueur. ", '10. AVANT BLESSURE OU MALADIE, je dis ou je fais des choses gênantes. ', "10. AUJOURD'HUI, je dis ou je fais des choses gênantes. ", "11. AVANT BLESSURE OU MALADIE, je n'églige mon hygiène personnelle. ", "11. AUJOURD'HUI, je n'églige mon hygiène personnelle. ", "12. AVANT BLESSURE OU MALADIE, je n'arrive pas à rester tranquille, je suis hyperactif (ve). ", "12. AUJOURD'HUI, je n'arrive pas à rester tranquille, je suis hyperactif (ve). ", '13. AVANT BLESSURE OU MALADIE, je ne me rends pas compte de mes problèmes ou de quand je fais une erreur. ', "13. AUJOURD'HUI, je ne me rends pas compte de mes problèmes ou de quand je fais une erreur. ", '14. AVANT BLESSURE OU MALADIE, je reste à ne rien faire. ', "14. AUJOURD'HUI, je reste à ne rien faire. ", '15. AVANT BLESSURE OU MALADIE, je suis désorganisé(e). ', "15. AUJOURD'HUI, je suis désorganisé(e). ", "16. AVANT BLESSURE OU MALADIE, je n'arrive pas à contrôler mon urine ou mes selles, mais ça ne me gêne pas. ", "16. AUJOURD'HUI, je n'arrive pas à contrôler mon urine ou mes selles, mais ça ne me gêne pas. ", '17. AVANT BLESSURE OU MALADIE, je ne peux pas faire deux choses à la fois (par exemple parler et préparer le repas). ', "17. AUJOURD'HUI, je ne peux pas faire deux choses à la fois (par exemple parler et préparer le repas). ", "18. AVANT BLESSURE OU MALADIE, je parle quand ce n'est pas à moi de le faire, j'interromps les gens dans les conversations. ", "18. AUJOURD'HUI, je parle quand ce n'est pas à moi de le faire, j'interromps les gens dans les conversations. ", "19. AVANT BLESSURE OU MALADIE, je n'ai pas beaucoup de discernement, je ne sais pas bien résoudre un problème. ", "19. AUJOURD'HUI, je n'ai pas beaucoup de discernement, je ne sais pas bien résoudre un problème. ", '20. AVANT BLESSURE OU MALADIE, je peux inventer des histoires extraordinaires, alors que je suis incapable de me souvenir de quelque chose. ', "20. AUJOURD'HUI, je peux inventer des histoires extraordinaires, alors que je suis incapable de me souvenir de quelque chose. ", "21. AVANT BLESSURE OU MALADIE, j'ai perdu l'intérêt pour des choses qui étaient auparavant amusantes ou importantes pour moi. ", "21. AUJOURD'HUI, j'ai perdu l'intérêt pour des choses qui étaient auparavant amusantes ou importantes pour moi. ", '22. AVANT BLESSURE OU MALADIE, je dis quelque chose, puis je fais autre chose. ', "22. AUJOURD'HUI, je dis quelque chose, puis je fais autre chose. ", "23. AVANT BLESSURE OU MALADIE, je commence des choses, mais je n'arrive pas à les finir, ça tombe à l'eau. ", "23. AUJOURD'HUI, je commence des choses, mais je n'arrive pas à les finir, ça tombe à l'eau. ", "24. AVANT BLESSURE OU MALADIE, je ne manifeste pas beaucoup d'émotions, je suis pas concerné(e) ou insensible. ", "24. AUJOURD'HUI, je ne manifeste pas beaucoup d'émotions, je suis pas concerné(e) ou insensible. ", "25. AVANT BLESSURE OU MALADIE, j'oublie de faire des choses, et puis je m'en souviens quand on me les rapelle ou quand il est trop tard. ", "25. AUJOURD'HUI, j'oublie de faire des choses, et puis je m'en souviens quand on me les rapelle ou quand il est trop tard. ", '26. AVANT BLESSURE OU MALADIE, je suis rigide, je suis incapable de changer mes habitudes. ', "26. AUJOURD'HUI, je suis rigide, je suis incapable de changer mes habitudes. ", "27. AVANT BLESSURE OU MALADIE, j'ai des ennuis avec la loi ou les autorités. ", "27. AUJOURD'HUI, j'ai des ennuis avec la loi ou les autorités. ", "28. AVANT BLESSURE OU MALADIE, je fais des choses dangeureuses juste pour m'amuser. ", "28. AUJOURD'HUI, je fais des choses dangeureuses juste pour m'amuser. ", "29. AVANT BLESSURE OU MALADIE, je me déplace lentement, je manque d'énergie, je suis inactif(ve). ", "29. AUJOURD'HUI, je me déplace lentement, je manque d'énergie, je suis inactif(ve). ", "30. AVANT BLESSURE OU MALADIE, je suis trop bête, j'ai un humour puéril.  ", "30. AUJOURD'HUI, je suis trop bête, j'ai un humour puéril.  ", "31. AVANT BLESSURE OU MALADIE, je trouve que la nourriture n'a pas de goût ni d'odeur. ", "31. AUJOURD'HUI, je trouve que la nourriture n'a pas de goût ni d'odeur. ", '32. AVANT BLESSURE OU MALADIE, je dis des gros mots. ', "32. AUJOURD'HUI, je dis des gros mots. ", "33. AVANT BLESSURE OU MALADIE, je m'excuse de mes écarts de comportement (par exemple, je m'excuse de dire des gros mots). ", "33. AUJOURD'HUI, je m'excuse de mes écarts de comportement (par exemple, je m'excuse de dire des gros mots). ", '34. AVANT BLESSURE OU MALADIE, je fais attention, je me concentre, même quand il y a des distractions. ', "34. AUJOURD'HUI, je fais attention, je me concentre, même quand il y a des distractions. ", "35. AVANT BLESSURE OU MALADIE, je réfléchis avant d'agir (par exemple, je pense au budget avant de dépenser de l'argent).  ", "35. AUJOURD'HUI, je réfléchis avant d'agir (par exemple, je pense au budget avant de dépenser de l'argent).  ", "36. AVANT BLESSURE OU MALADIE, j'utilise des stratégies pour me souvenir des choses importantes (par exemple, j'écris des notes pour moi-même). ", "36. AUJOURD'HUI, j'utilise des stratégies pour me souvenir des choses importantes (par exemple, j'écris des notes pour moi-même). ", '37. AVANT BLESSURE OU MALADIE, je suis capable de faire des projets. ', "37. AUJOURD'HUI, je suis capable de faire des projets. ", '38. AVANT BLESSURE OU MALADIE, je suis attiré(e) par le sexe. ', "38. AUJOURD'HUI, je suis attiré(e) par le sexe. ", '39. AVANT BLESSURE OU MALADIE, je fais attention à mon aspect extérieur (par exemple, je soigne ma présentation tous les jours). ', "39. AUJOURD'HUI, je fais attention à mon aspect extérieur (par exemple, je soigne ma présentation tous les jours). ", "40. AVANT BLESSURE OU MALADIE, je tire bénéfice des réactions que je provoque, j'accepte les critiques et les réactions des autres. ", "40. AUJOURD'HUI, je tire bénéfice des réactions que je provoque, j'accepte les critiques et les réactions des autres. ", '41. AVANT BLESSURE OU MALADIE, je me lance spontanément dans des activités ou des passe-temps. ', "41. AUJOURD'HUI, je me lance spontanément dans des activités ou des passe-temps. ", "42. AVANT BLESSURE OU MALADIE, je fais mes choses sans qu'on me le demande. ", "42. AUJOURD'HUI, je fais mes choses sans qu'on me le demande. ", "43. AVANT BLESSURE OU MALADIE, je suis sensible aux besoins d'autrui. ", "43. AUJOURD'HUI, je suis sensible aux besoins d'autrui. ", "44. AVANT BLESSURE OU MALADIE, je m'entends bien avec les autres gens. ", "44. AUJOURD'HUI, je m'entends bien avec les autres gens. ", "45. AVANT BLESSURE OU MALADIE, j'agis conformément à mon âge. ", "45. AUJOURD'HUI, j'agis conformément à mon âge. ", '46. AVANT BLESSURE OU MALADIE, je peux facilement engager une conversation. ', "46. AUJOURD'HUI, je peux facilement engager une conversation. ", 'Apathy Score (Before)', 'Apathy Score (After)', 'Disinhibition Score (Before)', 'Disinhibition Score (After)', 'Executive Dysfunction Score (Before)', 'Executive Dysfunction Score (After)']

In [25]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score

# ==========================================
# 1. MODEL ARCHITECTURE (CLS Token Version)
# ==========================================
class TabularTransformer(nn.Module):
    def __init__(self, column_vocab_sizes, num_numerical, num_classes, embed_dim=32, n_heads=4, n_layers=3):
        super(TabularTransformer, self).__init__()
        
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(v_size, embed_dim) for v_size in column_vocab_sizes
        ])
        
        self.num_projection = nn.Linear(num_numerical, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=n_heads, 
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x_cat, x_num):
        embeddings = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        x = torch.stack(embeddings, dim=1) 
        num_feat = self.num_projection(x_num).unsqueeze(1)
        
        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        
        x = torch.cat([cls_tokens, x, num_feat], dim=1)
        attn_out = self.transformer(x)
        cls_output = attn_out[:, 0, :] 
        return self.classifier(cls_output)

# ==========================================
# 2. ADVANCED DATA PREPARATION
# ==========================================
def find_col_by_prefix(df, prefix, length=30):
    prefix_fragment = prefix[:length].strip().lower()
    for col in df.columns:
        if col.strip().lower().startswith(prefix_fragment):
            return col
    return None

def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '.'), errors='coerce')

def prepare_data(df):
    # 1. IDENTIFY TARGET & DROP MISSING LABELS
    target_col = find_col_by_prefix(df, "Enrolment Group:", length=15)
    if not target_col: raise ValueError("Target column not found.")
    df = df.dropna(subset=[target_col]).copy()

    # 2. DEFINE MINIMAL FEATURE LISTS (Exact matches for your Word Doc)
    clinical_pairs = {
        #"blow": ("3. Have you ever experienced a hard blow to the head", "3a. How many times?"),
        "fall": ("15. Have you fallen in the last 3 months?", "15a. How many times did you fall?"),
        "dream": ("14. Have you ever been told or have you ever noticed that you were acting out your dreams?", "14a. Since when"),
        "constipation": ("5. Do you suffer from constipation?", "5a. How many years"),
        #"contact_sports": ("4. Have you ever played high contact sports?", "4a. How many years"),
        "sleep": ("12. Do you have trouble staying asleep?", "12a. Since when have you had trouble staying asleep?"),
        #"dozing_off": ("13. What are the chances", "13a. Since when"),
        "exercise": ("20. Do you exercise on a regular basis?", "20a. How often do you exercise per week on average?"),
        #"coffee": ("23. Do you currently drink coffee?", "23a. Since when"),
        "alcohol": ("24. Do you currently use any of the following? (Alcohol)", "24a. Since when"),
        "smoking": ("24h. (If you do NOT currently smoke cigarettes)", "24i. For how many years"),
        "dementia": ("22. Does the patient have dementia?", "Current duration of dementia"),
        "dbs": ("23. Has the patient had surgery for Parkinson's?", "Current duration of DBS"),
        "pd_dx": ("1. Was the patient diagnosed", "What is the current duration of disease")
    }

    standalone_cat = [
        "1. Gender/Genre:", 
        "2. What is your current marital status?",
        #"7. Do you have a regular caregiver? A caregiver as part",
        "1c. How was the diagnosis confirmed for C-OPN?",
        #"4. What were the first symptoms?",
        #"8. When the symptoms began, did the symptoms present on one side",
        #"9. What symptoms are currently present?",
        #"10. Do the symptoms affect both sides of your body now",
        "13. Do you currently have dyskinesia?",
        #"14. Do you experience freezing",
        "21. Hoehn & Yahr", 
        "2. Has the patient shown a significant reduction",
        #"3. Current COMT Inhibitor Medications",
        #"1. Have you ever been diagnosed with: ",
        "7. Do you ever have an irrepressible urge to move your legs?",
        "8. Do you feel or have people told you that your sense of smell is getting worse?",
        #"9. Have your friends or family told you lately that you are becoming more forgetful?",
        #"10. Have you noticed problems with your short term memory?",
        "18. Have you ever used or been exposed to pesticides?",
        "25. Does/did your biological father have Parkinson's disease?",
        #"25a. What is the ethnicity of your biological father? ",
        "26. Does/did your biological mother have Parkinson's disease?",
        #"26a. What is the ethnicity of your biological mother? ",
        #"18. Do you experience motor fluctuations?",
        #"Handedness Classification" # EHI
    ]

    continuous_num = [
        "Age at study visit", 
        #"What is the age at diagnosis?", 
        #"BMI", 
        #"What is the duration of disease at the time the clinical questionnaire was completed?",
        "Visuospatial/Executive Score",
        "Naming Score",
        "Attention Score",
        "Language Score",
        "Abstraction Score",
        "Delayed Recall Score",
        "Orientation Score",
        "TOTAL SCORE (make sure to include extra point",
        #"8. Total LED", 
        "Updrs_3_1 value", # speech
        "Updrs_3_2 value",
        "Updrs_3_3_neck value",
        "Updrs_3_3_rue value",
        "Updrs_3_3_lue value",
        "Updrs_3_3_rle value",
        "Updrs_3_3_lle value",
        "Updrs_3_4_r value", # finger tapping
        "Updrs_3_4_l value",
        "Updrs_3_5_r value", # hand movements
        "Updrs_3_5_l value",
        "Updrs_3_6_r value", # PRONATION-SUPINATION MOVEMENTS OF HANDS
        "Updrs_3_6_l value",
        "Updrs_3_7_r value", # 
        "Updrs_3_7_l value",
        "Updrs_3_8_r value", # 
        "Updrs_3_8_l value",
        "Updrs_3_9 value", # 
        "Updrs_3_10 value",
        "Updrs_3_11 value",
        "Updrs_3_12 value",
        "Updrs_3_13 value",
        "Updrs_3_14",
        "Updrs_3_15_r value",
        "Updrs_3_15_l value",
        "Updrs_3_16_r value",
        "Updrs_3_16_l value",
        "Updrs_3_17_rue value",	
        "Updrs_3_17_lue value",	
        "Updrs_3_17_rle value",	
        "Updrs_3_17_lle value",	
        "Updrs_3_17_lipjaw value",	
        "Updrs_3_18 value",
        "Part III: Motor Examination",
        "MBI-C Grand Total",
        "BDI-II Total Score",
        "BAI Total Score",
        #"EHI Total Score Right",
        #"EHI Total Score Left ",
        #"EHI Handedness Score ",
        "Command Clock z score",
        "Brixton z score",
        "Colour Words z score",
        "Trail A z score",
        "Trail B z score",
        #"Digit Span Forward z score",
        #"Digit Span Backward z score ",
        #"Digit Span Sequencing z score ",
        "Digit Span Total z score",
        #"FAS z score",
        "Animals z score ",
        #"Actions z score",
        "Actions Repetition z score",
        "Total (SR+SC) z score",
        "Total Recall z score",
        "Delayed Recall z score",
        #"Retention z score",
        "Recog z score",
        "Retention z score",
        "Memory z score",
        
    ]

    x_cat_list, x_num_list = [], []
    
    # 3. MECHANICAL DATA PROCESSING
    for key, (s_pref, d_pref) in clinical_pairs.items():
        cs, cd = find_col_by_prefix(df, s_pref), find_col_by_prefix(df, d_pref)
        assert cs and cd, f"Missing pair for {key}"
        # Force categorical to string to avoid NaN errors in LabelEncoder
        x_cat_list.append(LabelEncoder().fit_transform(df[cs].astype(str).fillna("Unknown")))
        # Durations: Fill missing years with 0
        x_num_list.append(clean_numeric(df[cd]).fillna(0).values)

    for pref in standalone_cat:
        col = find_col_by_prefix(df, pref)
        assert col, f"Missing categorical: {pref}"
        x_cat_list.append(LabelEncoder().fit_transform(df[col].astype(str).fillna("Unknown")))
    
    for pref in continuous_num:
        col = find_col_by_prefix(df, pref)
        assert col, f"Missing numerical: {pref}"
        num_col = clean_numeric(df[col])
        # Robust Imputation: Fill with median, fallback to 0 if column is entirely empty
        col_med = num_col.median()
        fill_val = col_med if not np.isnan(col_med) else 0
        x_num_list.append(num_col.fillna(fill_val).values)

    # 4. FINAL TENSOR CONSTRUCTION
    # Transpose and Scale
    x_num_final = StandardScaler().fit_transform(np.array(x_num_list).T)
    x_cat_final = np.array(x_cat_list).T
    
    # Verify No NaNs exist before returning
    if np.isnan(x_num_final).any():
        raise ValueError("CRITICAL: NaNs detected in numerical matrix after processing.")

    le = LabelEncoder()
    y = le.fit_transform(df[target_col])
    vocab_sizes = [int(x_cat_final[:, i].max()) + 1 for i in range(x_cat_final.shape[1])]
    
    return (torch.tensor(x_cat_final, dtype=torch.long), 
            torch.tensor(x_num_final, dtype=torch.float32), 
            torch.tensor(y, dtype=torch.long), 
            le.classes_, 
            vocab_sizes)


# ==========================================
# 3. EXECUTION WITH THRESHOLD OPTIMIZATION
# ==========================================
df_merged = pd.read_csv('/Users/jianyuebai/Desktop/Case Study/SSC_merged.csv', low_memory=False)
x_cat, x_num, y, classes, vocab_sizes = prepare_data(df_merged)

train_idx, val_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)
train_loader = DataLoader(TensorDataset(x_cat[train_idx], x_num[train_idx], y[train_idx]), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(x_cat[val_idx], x_num[val_idx], y[val_idx]), batch_size=16)

model = TabularTransformer(vocab_sizes, x_num.shape[1], len(classes))

class_weights = torch.tensor(1. / np.bincount(y), dtype=torch.float)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(30):
    model.train()
    for b_cat, b_num, b_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(b_cat, b_num), b_y)
        loss.backward()
        optimizer.step()

# --- FINAL EVALUATION WITH OPTIMIZED THRESHOLDS ---
model.eval()
all_probs, all_targets = [], []
with torch.no_grad():
    for b_cat, b_num, b_y in val_loader:
        logits = model(b_cat, b_num)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(b_y.cpu().numpy())

all_probs = np.vstack(all_probs)
all_targets = np.concatenate(all_targets)

best_thresholds = []
print("\n--- Optimized Thresholds per Class ---")

for i, class_name in enumerate(classes):
    # Binary targets for current class
    y_true_class = (all_targets == i).astype(int)
    y_prob_class = all_probs[:, i]
    
    # Calculate Precision-Recall Curve
    from sklearn.metrics import precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(y_true_class, y_prob_class)
    
    # Calculate F1 for each threshold
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    
    # thresholds array is 1 shorter than precision/recall
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_thresholds.append(best_threshold)
    
    class_auc = roc_auc_score(y_true_class, y_prob_class)
    print(f"{class_name:15} | Best Threshold: {best_threshold:.4f} | Max F1: {f1_scores[best_idx]:.4f} | AUC: {class_auc:.4f}")

# Apply optimized thresholds for final classification
final_preds = []
for i in range(len(all_probs)):
    # Calculate margin: how much each class prob exceeds its specific threshold
    margins = all_probs[i] - np.array(best_thresholds)
    final_preds.append(np.argmax(margins))

print("\nFinal Classification Report (Optimized Thresholds):")
print(classification_report(all_targets, final_preds, target_names=classes))

weighted_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='weighted')
print(f"Final Multi-class Weighted ROC-AUC: {weighted_auc:.4f}")


--- Optimized Thresholds per Class ---
AP (Atypical Parkinsonism)/(Parkinsonisme Atypique) | Best Threshold: 0.6874 | Max F1: 0.8750 | AUC: 0.9619
Healthy control/Contrôle | Best Threshold: 0.6448 | Max F1: 0.7143 | AUC: 0.9586
PD (Parkinson's Disease)/(Maladie de Parkinson) | Best Threshold: 0.2648 | Max F1: 0.9616 | AUC: 0.9596

Final Classification Report (Optimized Thresholds):
                                                     precision    recall  f1-score   support

AP (Atypical Parkinsonism)/(Parkinsonisme Atypique)       0.93      0.82      0.88        34
                           Healthy control/Contrôle       0.86      0.61      0.71        82
    PD (Parkinson's Disease)/(Maladie de Parkinson)       0.94      0.98      0.96       571

                                           accuracy                           0.93       687
                                          macro avg       0.91      0.81      0.85       687
                                       weighted avg   